# Copyright (C) 2026 Bruno Proença de Souza
# Licenciado sob GNU AGPL v3 - veja o arquivo LICENSE

## area_plantada_ha + váriaveis climáticas --> rendimento_kg_ha

In [8]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import RobustScaler, StandardScaler, PowerTransformer
from sklearn.model_selection import cross_val_score
import warnings
warnings.filterwarnings('ignore')

# ===============================================================================
# CONFIGURAÇÃO - ajuste apenas estes caminhos se necessário
# ===============================================================================
PARQUET_PATH  = r'C:\Users\bruno\Desktop\Pipeline_TCC\data\processed\dataset_final.parquet'
TARGET_COL    = 'rendimento_kg_ha'
YEAR_COL      = 'ano'
TRAIN_YEARS   = list(range(2018, 2023))   # 2018 a 2022 inclusive
TEST_YEARS    = [2023, 2024]
# ===============================================================================


class CriticalV1VariantsTest:
    """
    Teste focado em variações do critical_v1 com normalização otimizada.
    Lê um único arquivo parquet e faz o split por ano.
    """

    def __init__(self, parquet_path, target_col, year_col,
                 train_years, test_years):
        self.parquet_path = parquet_path
        self.target_col   = target_col
        self.year_col     = year_col
        self.train_years  = train_years
        self.test_years   = test_years
        self.results      = []

    # ------------------------------------------------------------------
    def load_and_clean_data(self):
        print("=" * 120)
        print("TESTE DE VARIAÇÕES DO CRITICAL_V1 - COM NORMALIZAÇÃO OTIMIZADA")
        print("=" * 120)

        df = pd.read_parquet(self.parquet_path)
        print(f"\n✓ Dataset carregado: {df.shape}  |  colunas: {df.columns.tolist()[:8]} …")

        # Split temporal
        self.train_df = df[df[self.year_col].isin(self.train_years)].dropna().reset_index(drop=True)
        self.test_df  = df[df[self.year_col].isin(self.test_years)].dropna().reset_index(drop=True)

        print(f"\n✓ Treino ({self.train_years[0]}–{self.train_years[-1]}): {self.train_df.shape} "
              f"| Teste ({self.test_years[0]}–{self.test_years[-1]}): {self.test_df.shape}")

        print(f"\n📊 TARGET ({self.target_col}) — ESTATÍSTICAS ORIGINAIS:")
        for label, subset in [("Treino", self.train_df), ("Teste ", self.test_df)]:
            s = subset[self.target_col]
            print(f"   {label}: média={s.mean():.2f}, std={s.std():.2f}, "
                  f"min={s.min():.2f}, max={s.max():.2f}")
        print()

    # ------------------------------------------------------------------
    def identify_climate_variables(self):
        """Detecta prefixos de variáveis climáticas do tipo <var>_dec<N>_ano<M>."""
        all_cols = self.train_df.columns.tolist()
        climate_vars = set()
        for col in all_cols:
            if 'dec' in col and 'ano' in col:
                var_name = col.split('dec')[0].rstrip('_')
                climate_vars.add(var_name)

        self.climate_vars = sorted(list(climate_vars))
        print(f"✓ {len(self.climate_vars)} variáveis climáticas identificadas: {self.climate_vars[:5]} …\n")

    # ------------------------------------------------------------------
    def get_variable_columns(self, var_name):
        return [col for col in self.train_df.columns
                if col.startswith(f"{var_name}_dec")]

    # ------------------------------------------------------------------
    def normalize_raw_data(self, train_df, test_df):
        """Normaliza colunas climáticas brutas ANTES da agregação."""
        print("🔧 Aplicando normalização nos dados brutos...")

        train_normalized = train_df.copy()
        test_normalized  = test_df.copy()

        climate_cols = [col for col in train_df.columns if 'dec' in col and 'ano' in col]

        for var in self.climate_vars:
            var_cols = [col for col in climate_cols if col.startswith(f"{var}_dec")]
            if var_cols:
                scaler = RobustScaler()
                train_normalized[var_cols] = scaler.fit_transform(train_df[var_cols])
                test_normalized[var_cols]  = scaler.transform(test_df[var_cols])

        print(f"   ✓ {len(climate_cols)} colunas climáticas normalizadas")
        return train_normalized, test_normalized

    # ------------------------------------------------------------------
    def aggregate_features(self, df, variant_config):
        """Agrega features por fase fenológica conforme configuração da variante."""
        non_climate_cols = [col for col in df.columns
                            if not any(col.startswith(f"{var}_dec")
                                       for var in self.climate_vars)]
        df_result = df[non_climate_cols].copy()

        for var in self.climate_vars:
            var_cols  = self.get_variable_columns(var)
            if not var_cols:
                continue

            ano1_cols = [c for c in var_cols if 'ano1' in c]
            ano2_cols = [c for c in var_cols if 'ano2' in c]

            # FASE 1: Pré-plantio (ano anterior)
            if variant_config['early']['enabled']:
                n = variant_config['early']['n_decendios']
                if ano1_cols and len(ano1_cols) >= n:
                    early_data = df[ano1_cols[-n:]]
                    cfg = variant_config['early']['stats']
                    if 'mean'   in cfg: df_result[f'{var}_early_mean']   = early_data.mean(axis=1)
                    if 'std'    in cfg: df_result[f'{var}_early_std']    = early_data.std(axis=1)
                    if 'min'    in cfg: df_result[f'{var}_early_min']    = early_data.min(axis=1)
                    if 'max'    in cfg: df_result[f'{var}_early_max']    = early_data.max(axis=1)

            # FASE 2: Florescimento
            fs, fe = variant_config['flowering']['start_dec'], variant_config['flowering']['end_dec']
            flowering_cols = [c for c in ano2_cols
                              if any(f'dec{d}_' in c for d in range(fs, fe + 1))]
            if flowering_cols:
                fd  = df[flowering_cols]
                cfg = variant_config['flowering']['stats']
                if 'mean'   in cfg: df_result[f'{var}_flowering_mean']   = fd.mean(axis=1)
                if 'std'    in cfg: df_result[f'{var}_flowering_std']    = fd.std(axis=1)
                if 'min'    in cfg: df_result[f'{var}_flowering_min']    = fd.min(axis=1)
                if 'max'    in cfg: df_result[f'{var}_flowering_max']    = fd.max(axis=1)
                if 'median' in cfg: df_result[f'{var}_flowering_median'] = fd.median(axis=1)

            # FASE 3: Enchimento de grãos
            gs, ge = variant_config['grain']['start_dec'], variant_config['grain']['end_dec']
            grain_cols = [c for c in ano2_cols
                          if any(f'dec{d}_' in c for d in range(gs, ge + 1))]
            if grain_cols:
                gd  = df[grain_cols]
                cfg = variant_config['grain']['stats']
                if 'mean'   in cfg: df_result[f'{var}_grain_mean']   = gd.mean(axis=1)
                if 'std'    in cfg: df_result[f'{var}_grain_std']    = gd.std(axis=1)
                if 'min'    in cfg: df_result[f'{var}_grain_min']    = gd.min(axis=1)
                if 'max'    in cfg: df_result[f'{var}_grain_max']    = gd.max(axis=1)
                if 'median' in cfg: df_result[f'{var}_grain_median'] = gd.median(axis=1)

            # FASE 4: Maturação (opcional)
            if variant_config['maturation']['enabled']:
                ms, me = variant_config['maturation']['start_dec'], variant_config['maturation']['end_dec']
                mat_cols = [c for c in ano2_cols
                            if any(f'dec{d}_' in c for d in range(ms, me + 1))]
                if mat_cols:
                    md  = df[mat_cols]
                    cfg = variant_config['maturation']['stats']
                    if 'mean' in cfg: df_result[f'{var}_maturation_mean'] = md.mean(axis=1)
                    if 'std'  in cfg: df_result[f'{var}_maturation_std']  = md.std(axis=1)

        return df_result

    # ------------------------------------------------------------------
    def select_features_rf(self, df_train, df_test, n_vars):
        X_train = df_train.drop(columns=[self.target_col])
        y_train = df_train[self.target_col]

        rf = RandomForestRegressor(n_estimators=100, max_depth=8,
                                   min_samples_leaf=5, random_state=42, n_jobs=-1)
        rf.fit(X_train, y_train)

        importances = pd.DataFrame({
            'feature': X_train.columns,
            'importance': rf.feature_importances_
        }).sort_values('importance', ascending=False)

        top_features = importances.head(n_vars)['feature'].tolist()
        return (df_train[top_features + [self.target_col]],
                df_test[top_features  + [self.target_col]],
                top_features)

    # ------------------------------------------------------------------
    def apply_scaling(self, X_train, X_test, scaler_type='robust'):
        if scaler_type == 'robust':
            scaler = RobustScaler()
        elif scaler_type == 'standard':
            scaler = StandardScaler()
        elif scaler_type == 'power':
            scaler = PowerTransformer(method='yeo-johnson', standardize=True)
        else:
            return X_train, X_test
        return scaler.fit_transform(X_train), scaler.transform(X_test)

    # ------------------------------------------------------------------
    def evaluate_with_cv(self, X, y, model, cv=5):
        scores = cross_val_score(model, X, y, cv=cv, scoring='r2', n_jobs=-1)
        return scores.mean(), scores.std()

    # ------------------------------------------------------------------
    def train_and_evaluate(self, df_train, df_test, variant_name,
                           n_features, model_config, scaler_type='robust'):
        X_train = df_train.drop(columns=[self.target_col])
        y_train = df_train[self.target_col]
        X_test  = df_test.drop(columns=[self.target_col])
        y_test  = df_test[self.target_col]

        X_train_s, X_test_s = self.apply_scaling(X_train, X_test, scaler_type)

        mtype  = model_config['type']
        params = model_config['params']

        if mtype == 'gbm':
            model       = GradientBoostingRegressor(**params, random_state=42)
            params_short = (f"{params['n_estimators']}e_d{params['max_depth']}"
                            f"_lr{params['learning_rate']}")
        elif mtype == 'svm':
            model       = SVR(**params)
            params_short = f"SVM_{params.get('kernel','rbf')}_C{params.get('C',1)}"
        elif mtype == 'linear':
            model       = LinearRegression(**params, n_jobs=-1)
            params_short = "Linear_OLS"
        else:   # rf
            model       = RandomForestRegressor(**params, random_state=42, n_jobs=-1)
            params_short = (f"{params['n_estimators']}e_d{params['max_depth']}"
                            f"_msl{params['min_samples_leaf']}")

        cv_mean, cv_std = self.evaluate_with_cv(X_train_s, y_train, model)
        model.fit(X_train_s, y_train)

        y_pred_train = model.predict(X_train_s)
        y_pred_test  = model.predict(X_test_s)

        r2_train = r2_score(y_train, y_pred_train)
        r2_test  = r2_score(y_test,  y_pred_test)

        result = {
            'variant':      variant_name,
            'n_features':   n_features,
            'model':        mtype,
            'scaler':       scaler_type,
            'params_short': params_short,
            'r2_train':     r2_train,
            'r2_test':      r2_test,
            'r2_cv':        cv_mean,
            'cv_std':       cv_std,
            'rmse_test':    np.sqrt(mean_squared_error(y_test, y_pred_test)),
            'mae_test':     mean_absolute_error(y_test, y_pred_test),
            'overfit':      r2_train - r2_test,
        }
        self.results.append(result)
        return result

    # ------------------------------------------------------------------
    def _drop_leakage_cols(self, df):
        """
        Remove colunas que vazam informação sobre o target ou são identificadores.
        Mantém 'ano' (pode ser preditor contextual leve) e 'area_plantada_ha'.
        Ajuste a lista conforme necessidade.
        """
        leakage_cols = [
            'quantidade_produzida_ton',
            'valor_producao_mil_reais',
            'valor_producao_pct',
            'valor_producao_ipca_mil_reais',
            'valor_producao_ipca_mil_reais_ha',
            'area_colhida_ha',
            'area_colhida_pct',
            'area_plantada_pct',
            'fator_correcao_ipca',
            # identificadores (não preditivos)
            'ano', 'cod_ibge', 'municipio', 'cod_meso', 'mesorregiao',
            'latitude', 'longitude',
        ]
        return df.drop(columns=[c for c in leakage_cols if c in df.columns])

    # ------------------------------------------------------------------
    def run_experiments(self):
        print("=" * 120)
        print("DEFINIÇÃO DAS VARIANTES")
        print("=" * 120 + "\n")

        # Remover colunas com leakage antes de qualquer processamento
        train_clean = self._drop_leakage_cols(self.train_df)
        test_clean  = self._drop_leakage_cols(self.test_df)

        # Normalizar dados brutos ANTES da agregação
        train_normalized, test_normalized = self.normalize_raw_data(train_clean, test_clean)

        variants = {
            'v1_original': {
                'name':        'V1: Original (3 dec early, 5-10 flow, 11-15 grain)',
                'early':       {'enabled': True,  'n_decendios': 3, 'stats': ['mean']},
                'flowering':   {'start_dec': 5,  'end_dec': 10, 'stats': ['mean']},
                'grain':       {'start_dec': 11, 'end_dec': 15, 'stats': ['mean']},
                'maturation':  {'enabled': False, 'start_dec': 16, 'end_dec': 18, 'stats': ['mean']},
            },
            'v6_with_variability': {
                'name':        'V6: Com variabilidade (mean + std)',
                'early':       {'enabled': True,  'n_decendios': 3, 'stats': ['mean', 'std']},
                'flowering':   {'start_dec': 5,  'end_dec': 10, 'stats': ['mean', 'std']},
                'grain':       {'start_dec': 11, 'end_dec': 15, 'stats': ['mean', 'std']},
                'maturation':  {'enabled': False, 'start_dec': 16, 'end_dec': 18, 'stats': ['mean']},
            },
            'v8_robust_stats': {
                'name':        'V8: Estatísticas robustas (mean + median)',
                'early':       {'enabled': True,  'n_decendios': 3, 'stats': ['mean']},
                'flowering':   {'start_dec': 5,  'end_dec': 10, 'stats': ['mean', 'median']},
                'grain':       {'start_dec': 11, 'end_dec': 15, 'stats': ['mean', 'median']},
                'maturation':  {'enabled': False, 'start_dec': 16, 'end_dec': 18, 'stats': ['mean']},
            },
            'v9_complete': {
                'name':        'V9: Completo (todas fases + variabilidade)',
                'early':       {'enabled': True,  'n_decendios': 4, 'stats': ['mean', 'std']},
                'flowering':   {'start_dec': 5,  'end_dec': 10, 'stats': ['mean', 'std']},
                'grain':       {'start_dec': 11, 'end_dec': 15, 'stats': ['mean', 'std']},
                'maturation':  {'enabled': True,  'start_dec': 16, 'end_dec': 18, 'stats': ['mean', 'std']},
            },
        }

        for key, cfg in variants.items():
            print(f"📋 {key}: {cfg['name']}")
        print()

        feature_counts = [35, 40, 45, 50]
        scalers        = ['robust', 'standard', 'power']
        model_configs  = [
            {'type': 'rf',     'params': {'n_estimators': 200, 'max_depth': 8,  'min_samples_split': 10, 'min_samples_leaf': 5}},
            {'type': 'rf',     'params': {'n_estimators': 250, 'max_depth': 8,  'min_samples_split': 10, 'min_samples_leaf': 5}},
            {'type': 'rf',     'params': {'n_estimators': 200, 'max_depth': 10, 'min_samples_split': 12, 'min_samples_leaf': 6}},
            {'type': 'gbm',    'params': {'n_estimators': 150, 'max_depth': 5,  'learning_rate': 0.03, 'subsample': 0.85}},
            {'type': 'gbm',    'params': {'n_estimators': 200, 'max_depth': 5,  'learning_rate': 0.02, 'subsample': 0.85}},
            {'type': 'svm',    'params': {'kernel': 'rbf',    'C': 10.0, 'epsilon': 0.1}},
            {'type': 'svm',    'params': {'kernel': 'linear', 'C': 1.0}},
            {'type': 'linear', 'params': {}},
        ]

        print("=" * 120)
        print("EXECUTANDO EXPERIMENTOS COM NORMALIZAÇÃO")
        print("=" * 120 + "\n")

        exp_num  = 0
        best_r2  = -float('inf')

        for var_key, var_config in variants.items():
            print(f"🔬 {var_config['name']}")

            train_agg = self.aggregate_features(train_normalized, var_config)
            test_agg  = self.aggregate_features(test_normalized,  var_config)

            n_features_available = train_agg.shape[1] - 1
            print(f"   Features disponíveis: {n_features_available}")

            for n_vars in feature_counts:
                if n_vars > n_features_available:
                    continue

                df_train_exp, df_test_exp, _ = self.select_features_rf(train_agg, test_agg, n_vars)

                for scaler_type in scalers:
                    for model_config in model_configs:
                        exp_num += 1
                        result   = self.train_and_evaluate(
                            df_train_exp, df_test_exp,
                            var_key, n_vars, model_config, scaler_type
                        )

                        if result['r2_test'] > best_r2:
                            best_r2 = result['r2_test']
                            print(f"   ⭐ [{exp_num:4d}] {model_config['type']} | {scaler_type:8s} | n={n_vars} | "
                                  f"R²Test={result['r2_test']:.4f} | R²CV={result['r2_cv']:.4f} | "
                                  f"Overfit={result['overfit']:.3f}")
                        elif result['r2_test'] > 0.28:
                            print(f"   ✓ [{exp_num:4d}] {model_config['type']} | {scaler_type:8s} | "
                                  f"n={n_vars} | R²={result['r2_test']:.4f}")
            print()

    # ------------------------------------------------------------------
    def display_results(self):
        print("=" * 120)
        print("🏆 TOP 30 RESULTADOS")
        print("=" * 120 + "\n")

        results_df = pd.DataFrame(self.results).sort_values('r2_test', ascending=False)

        print(f"{'#':<4} {'Variante':<22} {'n':<5} {'Modelo':<6} {'Scaler':<10} {'Config':<25} "
              f"{'R²Test':<9} {'R²CV':<9} {'Overfit':<8}")
        print("-" * 130)

        for idx, (_, row) in enumerate(results_df.head(30).iterrows(), 1):
            print(f"{idx:<4} {row['variant']:<22} {row['n_features']:<5} {row['model']:<6} "
                  f"{row['scaler']:<10} {row['params_short']:<25} "
                  f"{row['r2_test']:<9.4f} {row['r2_cv']:<9.4f} {row['overfit']:<8.3f}")

        best = results_df.iloc[0]

        print("\n" + "=" * 120)
        print("🎯 MELHOR CONFIGURAÇÃO COM NORMALIZAÇÃO")
        print("=" * 120)
        print(f"\nVariante: {best['variant']}")
        print(f"Features: {best['n_features']}")
        print(f"Modelo:   {best['model']}")
        print(f"Scaler:   {best['scaler']}")
        print(f"Config:   {best['params_short']}")
        print(f"\n📊 PERFORMANCE:")
        print(f"   R² Teste:    {best['r2_test']:.4f} ⭐")
        print(f"   R² CV:       {best['r2_cv']:.4f} ± {best['cv_std']:.4f}")
        print(f"   R² Treino:   {best['r2_train']:.4f}")
        print(f"   RMSE:        {best['rmse_test']:.2f} kg/ha")
        print(f"   MAE:         {best['mae_test']:.2f} kg/ha")
        print(f"   Overfitting: {best['overfit']:.4f}")

        print(f"\n📊 PERFORMANCE POR TIPO DE NORMALIZAÇÃO:")
        print("-" * 80)
        scaler_summary = results_df.groupby('scaler')['r2_test'].agg(['mean', 'max', 'count']).round(4)
        scaler_summary.columns = ['R² Médio', 'R² Máximo', 'Testes']
        print(scaler_summary.sort_values('R² Máximo', ascending=False))

        print(f"\n📊 PERFORMANCE POR VARIANTE:")
        print("-" * 80)
        variant_summary = results_df.groupby('variant')['r2_test'].agg(['mean', 'max', 'count']).round(4)
        variant_summary.columns = ['R² Médio', 'R² Máximo', 'Testes']
        print(variant_summary.sort_values('R² Máximo', ascending=False))

        print(f"\n📈 COMPARAÇÃO:")
        print(f"   Baseline (sem normalização):  R² = 0.2949")
        print(f"   Com normalização:             R² = {best['r2_test']:.4f}")
        improvement = (best['r2_test'] / 0.2949 - 1) * 100
        sign = '+' if improvement >= 0 else ''
        print(f"   Variação:                     {sign}{improvement:.2f}%")

        return results_df

    # ------------------------------------------------------------------
    def run(self):
        self.load_and_clean_data()
        self.identify_climate_variables()
        self.run_experiments()
        return self.display_results()


# ===============================================================================
# EXECUÇÃO
# ===============================================================================
if __name__ == "__main__":
    tester = CriticalV1VariantsTest(
        parquet_path = PARQUET_PATH,
        target_col   = TARGET_COL,
        year_col     = YEAR_COL,
        train_years  = TRAIN_YEARS,
        test_years   = TEST_YEARS,
    )
    results = tester.run()

    print("\n" + "=" * 120)
    print("✅ TESTE COM NORMALIZAÇÃO MELHORADA CONCLUÍDO!")
    print("=" * 120)

TESTE DE VARIAÇÕES DO CRITICAL_V1 - COM NORMALIZAÇÃO OTIMIZADA

✓ Dataset carregado: (2793, 8226)  |  colunas: ['cod_ibge', 'municipio', 'ano', 'cod_meso', 'mesorregiao', 'latitude', 'longitude', 'area_plantada_ha'] …

✓ Treino (2018–2022): (1995, 8226) | Teste (2023–2024): (798, 8226)

📊 TARGET (rendimento_kg_ha) — ESTATÍSTICAS ORIGINAIS:
   Treino: média=2968.33, std=1050.44, min=0.00, max=5400.00
   Teste : média=3276.66, std=813.62, min=0.00, max=5000.00

✓ 114 variáveis climáticas identificadas: ['AIRMASS', 'ALLSKY_KT', 'ALLSKY_NKT', 'ALLSKY_SFC_LW_DWN', 'ALLSKY_SFC_LW_UP'] …

DEFINIÇÃO DAS VARIANTES

🔧 Aplicando normalização nos dados brutos...
   ✓ 8208 colunas climáticas normalizadas
📋 v1_original: V1: Original (3 dec early, 5-10 flow, 11-15 grain)
📋 v6_with_variability: V6: Com variabilidade (mean + std)
📋 v8_robust_stats: V8: Estatísticas robustas (mean + median)
📋 v9_complete: V9: Completo (todas fases + variabilidade)

EXECUTANDO EXPERIMENTOS COM NORMALIZAÇÃO

🔬 V1: Origina

## area_plantada_ha + variáveis climáticas --> valor_producao_ha

In [9]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import RobustScaler, StandardScaler, PowerTransformer
from sklearn.model_selection import cross_val_score
import warnings
warnings.filterwarnings('ignore')

# ===============================================================================
# CONFIGURAÇÃO - ajuste apenas estes caminhos se necessário
# ===============================================================================
PARQUET_PATH  = r'C:\Users\bruno\Desktop\Pipeline_TCC\data\processed\dataset_final.parquet'
TARGET_COL    = 'valor_rs_ha'          # valor_producao_mil_reais / area_plantada_ha
YEAR_COL      = 'ano'
TRAIN_YEARS   = list(range(2018, 2023))   # 2018 a 2022 inclusive
TEST_YEARS    = [2023, 2024]
# ===============================================================================


class CriticalV1VariantsTest:
    """
    Teste focado em variações do critical_v1 com normalização otimizada.
    Lê um único arquivo parquet e faz o split por ano.
    """

    def __init__(self, parquet_path, target_col, year_col,
                 train_years, test_years):
        self.parquet_path = parquet_path
        self.target_col   = target_col
        self.year_col     = year_col
        self.train_years  = train_years
        self.test_years   = test_years
        self.results      = []

    # ------------------------------------------------------------------
    def load_and_clean_data(self):
        print("=" * 120)
        print("TESTE DE VARIAÇÕES DO CRITICAL_V1 - COM NORMALIZAÇÃO OTIMIZADA")
        print("=" * 120)

        df = pd.read_parquet(self.parquet_path)
        print(f"\n✓ Dataset carregado: {df.shape}  |  colunas: {df.columns.tolist()[:8]} …")

        # Criar target derivado: valor por hectare plantado (mil R$ → R$)
        df['valor_rs_ha'] = (df['valor_producao_mil_reais'] * 1_000) / df['area_plantada_ha']
        print("✓ Target 'valor_rs_ha' criado  (valor_producao_mil_reais × 1 000 / area_plantada_ha)")

        # Split temporal
        self.train_df = df[df[self.year_col].isin(self.train_years)].dropna().reset_index(drop=True)
        self.test_df  = df[df[self.year_col].isin(self.test_years)].dropna().reset_index(drop=True)

        print(f"\n✓ Treino ({self.train_years[0]}–{self.train_years[-1]}): {self.train_df.shape} "
              f"| Teste ({self.test_years[0]}–{self.test_years[-1]}): {self.test_df.shape}")

        print(f"\n📊 TARGET ({self.target_col}) — ESTATÍSTICAS ORIGINAIS:")
        for label, subset in [("Treino", self.train_df), ("Teste ", self.test_df)]:
            s = subset[self.target_col]
            print(f"   {label}: média={s.mean():.2f}, std={s.std():.2f}, "
                  f"min={s.min():.2f}, max={s.max():.2f}")
        print()

    # ------------------------------------------------------------------
    def identify_climate_variables(self):
        """Detecta prefixos de variáveis climáticas do tipo <var>_dec<N>_ano<M>."""
        all_cols = self.train_df.columns.tolist()
        climate_vars = set()
        for col in all_cols:
            if 'dec' in col and 'ano' in col:
                var_name = col.split('dec')[0].rstrip('_')
                climate_vars.add(var_name)

        self.climate_vars = sorted(list(climate_vars))
        print(f"✓ {len(self.climate_vars)} variáveis climáticas identificadas: {self.climate_vars[:5]} …\n")

    # ------------------------------------------------------------------
    def get_variable_columns(self, var_name):
        return [col for col in self.train_df.columns
                if col.startswith(f"{var_name}_dec")]

    # ------------------------------------------------------------------
    def normalize_raw_data(self, train_df, test_df):
        """Normaliza colunas climáticas brutas ANTES da agregação."""
        print("🔧 Aplicando normalização nos dados brutos...")

        train_normalized = train_df.copy()
        test_normalized  = test_df.copy()

        climate_cols = [col for col in train_df.columns if 'dec' in col and 'ano' in col]

        for var in self.climate_vars:
            var_cols = [col for col in climate_cols if col.startswith(f"{var}_dec")]
            if var_cols:
                scaler = RobustScaler()
                train_normalized[var_cols] = scaler.fit_transform(train_df[var_cols])
                test_normalized[var_cols]  = scaler.transform(test_df[var_cols])

        print(f"   ✓ {len(climate_cols)} colunas climáticas normalizadas")
        return train_normalized, test_normalized

    # ------------------------------------------------------------------
    def aggregate_features(self, df, variant_config):
        """Agrega features por fase fenológica conforme configuração da variante."""
        non_climate_cols = [col for col in df.columns
                            if not any(col.startswith(f"{var}_dec")
                                       for var in self.climate_vars)]
        df_result = df[non_climate_cols].copy()

        for var in self.climate_vars:
            var_cols  = self.get_variable_columns(var)
            if not var_cols:
                continue

            ano1_cols = [c for c in var_cols if 'ano1' in c]
            ano2_cols = [c for c in var_cols if 'ano2' in c]

            # FASE 1: Pré-plantio (ano anterior)
            if variant_config['early']['enabled']:
                n = variant_config['early']['n_decendios']
                if ano1_cols and len(ano1_cols) >= n:
                    early_data = df[ano1_cols[-n:]]
                    cfg = variant_config['early']['stats']
                    if 'mean'   in cfg: df_result[f'{var}_early_mean']   = early_data.mean(axis=1)
                    if 'std'    in cfg: df_result[f'{var}_early_std']    = early_data.std(axis=1)
                    if 'min'    in cfg: df_result[f'{var}_early_min']    = early_data.min(axis=1)
                    if 'max'    in cfg: df_result[f'{var}_early_max']    = early_data.max(axis=1)

            # FASE 2: Florescimento
            fs, fe = variant_config['flowering']['start_dec'], variant_config['flowering']['end_dec']
            flowering_cols = [c for c in ano2_cols
                              if any(f'dec{d}_' in c for d in range(fs, fe + 1))]
            if flowering_cols:
                fd  = df[flowering_cols]
                cfg = variant_config['flowering']['stats']
                if 'mean'   in cfg: df_result[f'{var}_flowering_mean']   = fd.mean(axis=1)
                if 'std'    in cfg: df_result[f'{var}_flowering_std']    = fd.std(axis=1)
                if 'min'    in cfg: df_result[f'{var}_flowering_min']    = fd.min(axis=1)
                if 'max'    in cfg: df_result[f'{var}_flowering_max']    = fd.max(axis=1)
                if 'median' in cfg: df_result[f'{var}_flowering_median'] = fd.median(axis=1)

            # FASE 3: Enchimento de grãos
            gs, ge = variant_config['grain']['start_dec'], variant_config['grain']['end_dec']
            grain_cols = [c for c in ano2_cols
                          if any(f'dec{d}_' in c for d in range(gs, ge + 1))]
            if grain_cols:
                gd  = df[grain_cols]
                cfg = variant_config['grain']['stats']
                if 'mean'   in cfg: df_result[f'{var}_grain_mean']   = gd.mean(axis=1)
                if 'std'    in cfg: df_result[f'{var}_grain_std']    = gd.std(axis=1)
                if 'min'    in cfg: df_result[f'{var}_grain_min']    = gd.min(axis=1)
                if 'max'    in cfg: df_result[f'{var}_grain_max']    = gd.max(axis=1)
                if 'median' in cfg: df_result[f'{var}_grain_median'] = gd.median(axis=1)

            # FASE 4: Maturação (opcional)
            if variant_config['maturation']['enabled']:
                ms, me = variant_config['maturation']['start_dec'], variant_config['maturation']['end_dec']
                mat_cols = [c for c in ano2_cols
                            if any(f'dec{d}_' in c for d in range(ms, me + 1))]
                if mat_cols:
                    md  = df[mat_cols]
                    cfg = variant_config['maturation']['stats']
                    if 'mean' in cfg: df_result[f'{var}_maturation_mean'] = md.mean(axis=1)
                    if 'std'  in cfg: df_result[f'{var}_maturation_std']  = md.std(axis=1)

        return df_result

    # ------------------------------------------------------------------
    def select_features_rf(self, df_train, df_test, n_vars):
        X_train = df_train.drop(columns=[self.target_col])
        y_train = df_train[self.target_col]

        rf = RandomForestRegressor(n_estimators=100, max_depth=8,
                                   min_samples_leaf=5, random_state=42, n_jobs=-1)
        rf.fit(X_train, y_train)

        importances = pd.DataFrame({
            'feature': X_train.columns,
            'importance': rf.feature_importances_
        }).sort_values('importance', ascending=False)

        top_features = importances.head(n_vars)['feature'].tolist()
        return (df_train[top_features + [self.target_col]],
                df_test[top_features  + [self.target_col]],
                top_features)

    # ------------------------------------------------------------------
    def apply_scaling(self, X_train, X_test, scaler_type='robust'):
        if scaler_type == 'robust':
            scaler = RobustScaler()
        elif scaler_type == 'standard':
            scaler = StandardScaler()
        elif scaler_type == 'power':
            scaler = PowerTransformer(method='yeo-johnson', standardize=True)
        else:
            return X_train, X_test
        return scaler.fit_transform(X_train), scaler.transform(X_test)

    # ------------------------------------------------------------------
    def evaluate_with_cv(self, X, y, model, cv=5):
        scores = cross_val_score(model, X, y, cv=cv, scoring='r2', n_jobs=-1)
        return scores.mean(), scores.std()

    # ------------------------------------------------------------------
    def train_and_evaluate(self, df_train, df_test, variant_name,
                           n_features, model_config, scaler_type='robust'):
        X_train = df_train.drop(columns=[self.target_col])
        y_train = df_train[self.target_col]
        X_test  = df_test.drop(columns=[self.target_col])
        y_test  = df_test[self.target_col]

        X_train_s, X_test_s = self.apply_scaling(X_train, X_test, scaler_type)

        mtype  = model_config['type']
        params = model_config['params']

        if mtype == 'gbm':
            model       = GradientBoostingRegressor(**params, random_state=42)
            params_short = (f"{params['n_estimators']}e_d{params['max_depth']}"
                            f"_lr{params['learning_rate']}")
        elif mtype == 'svm':
            model       = SVR(**params)
            params_short = f"SVM_{params.get('kernel','rbf')}_C{params.get('C',1)}"
        elif mtype == 'linear':
            model       = LinearRegression(**params, n_jobs=-1)
            params_short = "Linear_OLS"
        else:   # rf
            model       = RandomForestRegressor(**params, random_state=42, n_jobs=-1)
            params_short = (f"{params['n_estimators']}e_d{params['max_depth']}"
                            f"_msl{params['min_samples_leaf']}")

        cv_mean, cv_std = self.evaluate_with_cv(X_train_s, y_train, model)
        model.fit(X_train_s, y_train)

        y_pred_train = model.predict(X_train_s)
        y_pred_test  = model.predict(X_test_s)

        r2_train = r2_score(y_train, y_pred_train)
        r2_test  = r2_score(y_test,  y_pred_test)

        result = {
            'variant':      variant_name,
            'n_features':   n_features,
            'model':        mtype,
            'scaler':       scaler_type,
            'params_short': params_short,
            'r2_train':     r2_train,
            'r2_test':      r2_test,
            'r2_cv':        cv_mean,
            'cv_std':       cv_std,
            'rmse_test':    np.sqrt(mean_squared_error(y_test, y_pred_test)),
            'mae_test':     mean_absolute_error(y_test, y_pred_test),
            'overfit':      r2_train - r2_test,
        }
        self.results.append(result)
        return result

    # ------------------------------------------------------------------
    def _drop_leakage_cols(self, df):
        """
        Remove colunas que vazam informação sobre o target ou são identificadores.
        Mantém 'ano' (pode ser preditor contextual leve) e 'area_plantada_ha'.
        Ajuste a lista conforme necessidade.
        """
        leakage_cols = [
            # colunas que derivam diretamente do target ou causam leakage
            'rendimento_kg_ha',
            'quantidade_produzida_ton',
            'valor_producao_mil_reais',      # componente do target → leakage direto
            'valor_producao_pct',
            'valor_producao_ipca_mil_reais',
            'valor_producao_ipca_mil_reais_ha',
            'area_colhida_ha',
            'area_colhida_pct',
            'area_plantada_pct',
            'fator_correcao_ipca',
            # identificadores (não preditivos)
            'ano', 'cod_ibge', 'municipio', 'cod_meso', 'mesorregiao',
            'latitude', 'longitude',
        ]
        return df.drop(columns=[c for c in leakage_cols if c in df.columns])

    # ------------------------------------------------------------------
    def run_experiments(self):
        print("=" * 120)
        print("DEFINIÇÃO DAS VARIANTES")
        print("=" * 120 + "\n")

        # Remover colunas com leakage antes de qualquer processamento
        train_clean = self._drop_leakage_cols(self.train_df)
        test_clean  = self._drop_leakage_cols(self.test_df)

        # Normalizar dados brutos ANTES da agregação
        train_normalized, test_normalized = self.normalize_raw_data(train_clean, test_clean)

        variants = {
            'v1_original': {
                'name':        'V1: Original (3 dec early, 5-10 flow, 11-15 grain)',
                'early':       {'enabled': True,  'n_decendios': 3, 'stats': ['mean']},
                'flowering':   {'start_dec': 5,  'end_dec': 10, 'stats': ['mean']},
                'grain':       {'start_dec': 11, 'end_dec': 15, 'stats': ['mean']},
                'maturation':  {'enabled': False, 'start_dec': 16, 'end_dec': 18, 'stats': ['mean']},
            },
            'v6_with_variability': {
                'name':        'V6: Com variabilidade (mean + std)',
                'early':       {'enabled': True,  'n_decendios': 3, 'stats': ['mean', 'std']},
                'flowering':   {'start_dec': 5,  'end_dec': 10, 'stats': ['mean', 'std']},
                'grain':       {'start_dec': 11, 'end_dec': 15, 'stats': ['mean', 'std']},
                'maturation':  {'enabled': False, 'start_dec': 16, 'end_dec': 18, 'stats': ['mean']},
            },
            'v8_robust_stats': {
                'name':        'V8: Estatísticas robustas (mean + median)',
                'early':       {'enabled': True,  'n_decendios': 3, 'stats': ['mean']},
                'flowering':   {'start_dec': 5,  'end_dec': 10, 'stats': ['mean', 'median']},
                'grain':       {'start_dec': 11, 'end_dec': 15, 'stats': ['mean', 'median']},
                'maturation':  {'enabled': False, 'start_dec': 16, 'end_dec': 18, 'stats': ['mean']},
            },
            'v9_complete': {
                'name':        'V9: Completo (todas fases + variabilidade)',
                'early':       {'enabled': True,  'n_decendios': 4, 'stats': ['mean', 'std']},
                'flowering':   {'start_dec': 5,  'end_dec': 10, 'stats': ['mean', 'std']},
                'grain':       {'start_dec': 11, 'end_dec': 15, 'stats': ['mean', 'std']},
                'maturation':  {'enabled': True,  'start_dec': 16, 'end_dec': 18, 'stats': ['mean', 'std']},
            },
        }

        for key, cfg in variants.items():
            print(f"📋 {key}: {cfg['name']}")
        print()

        feature_counts = [35, 40, 45, 50]
        scalers        = ['robust', 'standard', 'power']
        model_configs  = [
            {'type': 'rf',     'params': {'n_estimators': 200, 'max_depth': 8,  'min_samples_split': 10, 'min_samples_leaf': 5}},
            {'type': 'rf',     'params': {'n_estimators': 250, 'max_depth': 8,  'min_samples_split': 10, 'min_samples_leaf': 5}},
            {'type': 'rf',     'params': {'n_estimators': 200, 'max_depth': 10, 'min_samples_split': 12, 'min_samples_leaf': 6}},
            {'type': 'gbm',    'params': {'n_estimators': 150, 'max_depth': 5,  'learning_rate': 0.03, 'subsample': 0.85}},
            {'type': 'gbm',    'params': {'n_estimators': 200, 'max_depth': 5,  'learning_rate': 0.02, 'subsample': 0.85}},
            {'type': 'svm',    'params': {'kernel': 'rbf',    'C': 10.0, 'epsilon': 0.1}},
            {'type': 'svm',    'params': {'kernel': 'linear', 'C': 1.0}},
            {'type': 'linear', 'params': {}},
        ]

        print("=" * 120)
        print("EXECUTANDO EXPERIMENTOS COM NORMALIZAÇÃO")
        print("=" * 120 + "\n")

        exp_num  = 0
        best_r2  = -float('inf')

        for var_key, var_config in variants.items():
            print(f"🔬 {var_config['name']}")

            train_agg = self.aggregate_features(train_normalized, var_config)
            test_agg  = self.aggregate_features(test_normalized,  var_config)

            n_features_available = train_agg.shape[1] - 1
            print(f"   Features disponíveis: {n_features_available}")

            for n_vars in feature_counts:
                if n_vars > n_features_available:
                    continue

                df_train_exp, df_test_exp, _ = self.select_features_rf(train_agg, test_agg, n_vars)

                for scaler_type in scalers:
                    for model_config in model_configs:
                        exp_num += 1
                        result   = self.train_and_evaluate(
                            df_train_exp, df_test_exp,
                            var_key, n_vars, model_config, scaler_type
                        )

                        if result['r2_test'] > best_r2:
                            best_r2 = result['r2_test']
                            print(f"   ⭐ [{exp_num:4d}] {model_config['type']} | {scaler_type:8s} | n={n_vars} | "
                                  f"R²Test={result['r2_test']:.4f} | R²CV={result['r2_cv']:.4f} | "
                                  f"Overfit={result['overfit']:.3f}")
                        elif result['r2_test'] > 0.28:
                            print(f"   ✓ [{exp_num:4d}] {model_config['type']} | {scaler_type:8s} | "
                                  f"n={n_vars} | R²={result['r2_test']:.4f}")
            print()

    # ------------------------------------------------------------------
    def display_results(self):
        print("=" * 120)
        print("🏆 TOP 30 RESULTADOS")
        print("=" * 120 + "\n")

        results_df = pd.DataFrame(self.results).sort_values('r2_test', ascending=False)

        print(f"{'#':<4} {'Variante':<22} {'n':<5} {'Modelo':<6} {'Scaler':<10} {'Config':<25} "
              f"{'R²Test':<9} {'R²CV':<9} {'Overfit':<8}")
        print("-" * 130)

        for idx, (_, row) in enumerate(results_df.head(30).iterrows(), 1):
            print(f"{idx:<4} {row['variant']:<22} {row['n_features']:<5} {row['model']:<6} "
                  f"{row['scaler']:<10} {row['params_short']:<25} "
                  f"{row['r2_test']:<9.4f} {row['r2_cv']:<9.4f} {row['overfit']:<8.3f}")

        best = results_df.iloc[0]

        print("\n" + "=" * 120)
        print("🎯 MELHOR CONFIGURAÇÃO COM NORMALIZAÇÃO")
        print("=" * 120)
        print(f"\nVariante: {best['variant']}")
        print(f"Features: {best['n_features']}")
        print(f"Modelo:   {best['model']}")
        print(f"Scaler:   {best['scaler']}")
        print(f"Config:   {best['params_short']}")
        print(f"\n📊 PERFORMANCE:")
        print(f"   R² Teste:    {best['r2_test']:.4f} ⭐")
        print(f"   R² CV:       {best['r2_cv']:.4f} ± {best['cv_std']:.4f}")
        print(f"   R² Treino:   {best['r2_train']:.4f}")
        print(f"   RMSE:        {best['rmse_test']:.2f} mil R$ (IPCA)")
        print(f"   MAE:         {best['mae_test']:.2f} mil R$ (IPCA)")
        print(f"   Overfitting: {best['overfit']:.4f}")

        print(f"\n📊 PERFORMANCE POR TIPO DE NORMALIZAÇÃO:")
        print("-" * 80)
        scaler_summary = results_df.groupby('scaler')['r2_test'].agg(['mean', 'max', 'count']).round(4)
        scaler_summary.columns = ['R² Médio', 'R² Máximo', 'Testes']
        print(scaler_summary.sort_values('R² Máximo', ascending=False))

        print(f"\n📊 PERFORMANCE POR VARIANTE:")
        print("-" * 80)
        variant_summary = results_df.groupby('variant')['r2_test'].agg(['mean', 'max', 'count']).round(4)
        variant_summary.columns = ['R² Médio', 'R² Máximo', 'Testes']
        print(variant_summary.sort_values('R² Máximo', ascending=False))

        print(f"\n📈 COMPARAÇÃO:")
        print(f"   Baseline (sem normalização):  R² = 0.2949")
        print(f"   Com normalização:             R² = {best['r2_test']:.4f}")
        improvement = (best['r2_test'] / 0.2949 - 1) * 100
        sign = '+' if improvement >= 0 else ''
        print(f"   Variação:                     {sign}{improvement:.2f}%")

        return results_df

    # ------------------------------------------------------------------
    def run(self):
        self.load_and_clean_data()
        self.identify_climate_variables()
        self.run_experiments()
        return self.display_results()


# ===============================================================================
# EXECUÇÃO
# ===============================================================================
if __name__ == "__main__":
    tester = CriticalV1VariantsTest(
        parquet_path = PARQUET_PATH,
        target_col   = TARGET_COL,
        year_col     = YEAR_COL,
        train_years  = TRAIN_YEARS,
        test_years   = TEST_YEARS,
    )
    results = tester.run()

    print("\n" + "=" * 120)
    print("✅ TESTE COM NORMALIZAÇÃO MELHORADA CONCLUÍDO!")
    print("=" * 120)

TESTE DE VARIAÇÕES DO CRITICAL_V1 - COM NORMALIZAÇÃO OTIMIZADA

✓ Dataset carregado: (2793, 8226)  |  colunas: ['cod_ibge', 'municipio', 'ano', 'cod_meso', 'mesorregiao', 'latitude', 'longitude', 'area_plantada_ha'] …
✓ Target 'valor_rs_ha' criado  (valor_producao_mil_reais × 1 000 / area_plantada_ha)

✓ Treino (2018–2022): (1910, 8227) | Teste (2023–2024): (780, 8227)

📊 TARGET (valor_rs_ha) — ESTATÍSTICAS ORIGINAIS:
   Treino: média=5480.42, std=2616.61, min=606.85, max=12600.00
   Teste : média=7196.69, std=2038.40, min=0.00, max=12656.76

✓ 114 variáveis climáticas identificadas: ['AIRMASS', 'ALLSKY_KT', 'ALLSKY_NKT', 'ALLSKY_SFC_LW_DWN', 'ALLSKY_SFC_LW_UP'] …

DEFINIÇÃO DAS VARIANTES

🔧 Aplicando normalização nos dados brutos...
   ✓ 8208 colunas climáticas normalizadas
📋 v1_original: V1: Original (3 dec early, 5-10 flow, 11-15 grain)
📋 v6_with_variability: V6: Com variabilidade (mean + std)
📋 v8_robust_stats: V8: Estatísticas robustas (mean + median)
📋 v9_complete: V9: Completo (

area_plantada_ha + features climáticas agregadas  --> quantidade_produzida_ton

In [10]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import RobustScaler, StandardScaler, PowerTransformer
from sklearn.model_selection import cross_val_score
import warnings
warnings.filterwarnings('ignore')

# ===============================================================================
# CONFIGURAÇÃO - ajuste apenas estes caminhos se necessário
# ===============================================================================
PARQUET_PATH  = r'C:\Users\bruno\Desktop\Pipeline_TCC\data\processed\dataset_final.parquet'
TARGET_COL    = 'quantidade_produzida_ton'
YEAR_COL      = 'ano'
TRAIN_YEARS   = list(range(2018, 2023))   # 2018 a 2022 inclusive
TEST_YEARS    = [2023, 2024]
# ===============================================================================


class CriticalV1VariantsTest:
    """
    Teste focado em variações do critical_v1 com normalização otimizada.
    Lê um único arquivo parquet e faz o split por ano.
    """

    def __init__(self, parquet_path, target_col, year_col,
                 train_years, test_years):
        self.parquet_path = parquet_path
        self.target_col   = target_col
        self.year_col     = year_col
        self.train_years  = train_years
        self.test_years   = test_years
        self.results      = []

    # ------------------------------------------------------------------
    def load_and_clean_data(self):
        print("=" * 120)
        print("TESTE DE VARIAÇÕES DO CRITICAL_V1 - COM NORMALIZAÇÃO OTIMIZADA")
        print("=" * 120)

        df = pd.read_parquet(self.parquet_path)
        print(f"\n✓ Dataset carregado: {df.shape}  |  colunas: {df.columns.tolist()[:8]} …")

        # Split temporal
        self.train_df = df[df[self.year_col].isin(self.train_years)].dropna().reset_index(drop=True)
        self.test_df  = df[df[self.year_col].isin(self.test_years)].dropna().reset_index(drop=True)

        print(f"\n✓ Treino ({self.train_years[0]}–{self.train_years[-1]}): {self.train_df.shape} "
              f"| Teste ({self.test_years[0]}–{self.test_years[-1]}): {self.test_df.shape}")

        print(f"\n📊 TARGET ({self.target_col}) — ESTATÍSTICAS ORIGINAIS:")
        for label, subset in [("Treino", self.train_df), ("Teste ", self.test_df)]:
            s = subset[self.target_col]
            print(f"   {label}: média={s.mean():.2f}, std={s.std():.2f}, "
                  f"min={s.min():.2f}, max={s.max():.2f}")
        print()

    # ------------------------------------------------------------------
    def identify_climate_variables(self):
        """Detecta prefixos de variáveis climáticas do tipo <var>_dec<N>_ano<M>."""
        all_cols = self.train_df.columns.tolist()
        climate_vars = set()
        for col in all_cols:
            if 'dec' in col and 'ano' in col:
                var_name = col.split('dec')[0].rstrip('_')
                climate_vars.add(var_name)

        self.climate_vars = sorted(list(climate_vars))
        print(f"✓ {len(self.climate_vars)} variáveis climáticas identificadas: {self.climate_vars[:5]} …\n")

    # ------------------------------------------------------------------
    def get_variable_columns(self, var_name):
        return [col for col in self.train_df.columns
                if col.startswith(f"{var_name}_dec")]

    # ------------------------------------------------------------------
    def normalize_raw_data(self, train_df, test_df):
        """Normaliza colunas climáticas brutas ANTES da agregação."""
        print("🔧 Aplicando normalização nos dados brutos...")

        train_normalized = train_df.copy()
        test_normalized  = test_df.copy()

        climate_cols = [col for col in train_df.columns if 'dec' in col and 'ano' in col]

        for var in self.climate_vars:
            var_cols = [col for col in climate_cols if col.startswith(f"{var}_dec")]
            if var_cols:
                scaler = RobustScaler()
                train_normalized[var_cols] = scaler.fit_transform(train_df[var_cols])
                test_normalized[var_cols]  = scaler.transform(test_df[var_cols])

        print(f"   ✓ {len(climate_cols)} colunas climáticas normalizadas")
        return train_normalized, test_normalized

    # ------------------------------------------------------------------
    def aggregate_features(self, df, variant_config):
        """Agrega features por fase fenológica conforme configuração da variante."""
        non_climate_cols = [col for col in df.columns
                            if not any(col.startswith(f"{var}_dec")
                                       for var in self.climate_vars)]
        df_result = df[non_climate_cols].copy()

        for var in self.climate_vars:
            var_cols  = self.get_variable_columns(var)
            if not var_cols:
                continue

            ano1_cols = [c for c in var_cols if 'ano1' in c]
            ano2_cols = [c for c in var_cols if 'ano2' in c]

            # FASE 1: Pré-plantio (ano anterior)
            if variant_config['early']['enabled']:
                n = variant_config['early']['n_decendios']
                if ano1_cols and len(ano1_cols) >= n:
                    early_data = df[ano1_cols[-n:]]
                    cfg = variant_config['early']['stats']
                    if 'mean'   in cfg: df_result[f'{var}_early_mean']   = early_data.mean(axis=1)
                    if 'std'    in cfg: df_result[f'{var}_early_std']    = early_data.std(axis=1)
                    if 'min'    in cfg: df_result[f'{var}_early_min']    = early_data.min(axis=1)
                    if 'max'    in cfg: df_result[f'{var}_early_max']    = early_data.max(axis=1)

            # FASE 2: Florescimento
            fs, fe = variant_config['flowering']['start_dec'], variant_config['flowering']['end_dec']
            flowering_cols = [c for c in ano2_cols
                              if any(f'dec{d}_' in c for d in range(fs, fe + 1))]
            if flowering_cols:
                fd  = df[flowering_cols]
                cfg = variant_config['flowering']['stats']
                if 'mean'   in cfg: df_result[f'{var}_flowering_mean']   = fd.mean(axis=1)
                if 'std'    in cfg: df_result[f'{var}_flowering_std']    = fd.std(axis=1)
                if 'min'    in cfg: df_result[f'{var}_flowering_min']    = fd.min(axis=1)
                if 'max'    in cfg: df_result[f'{var}_flowering_max']    = fd.max(axis=1)
                if 'median' in cfg: df_result[f'{var}_flowering_median'] = fd.median(axis=1)

            # FASE 3: Enchimento de grãos
            gs, ge = variant_config['grain']['start_dec'], variant_config['grain']['end_dec']
            grain_cols = [c for c in ano2_cols
                          if any(f'dec{d}_' in c for d in range(gs, ge + 1))]
            if grain_cols:
                gd  = df[grain_cols]
                cfg = variant_config['grain']['stats']
                if 'mean'   in cfg: df_result[f'{var}_grain_mean']   = gd.mean(axis=1)
                if 'std'    in cfg: df_result[f'{var}_grain_std']    = gd.std(axis=1)
                if 'min'    in cfg: df_result[f'{var}_grain_min']    = gd.min(axis=1)
                if 'max'    in cfg: df_result[f'{var}_grain_max']    = gd.max(axis=1)
                if 'median' in cfg: df_result[f'{var}_grain_median'] = gd.median(axis=1)

            # FASE 4: Maturação (opcional)
            if variant_config['maturation']['enabled']:
                ms, me = variant_config['maturation']['start_dec'], variant_config['maturation']['end_dec']
                mat_cols = [c for c in ano2_cols
                            if any(f'dec{d}_' in c for d in range(ms, me + 1))]
                if mat_cols:
                    md  = df[mat_cols]
                    cfg = variant_config['maturation']['stats']
                    if 'mean' in cfg: df_result[f'{var}_maturation_mean'] = md.mean(axis=1)
                    if 'std'  in cfg: df_result[f'{var}_maturation_std']  = md.std(axis=1)

        return df_result

    # ------------------------------------------------------------------
    def select_features_rf(self, df_train, df_test, n_vars):
        X_train = df_train.drop(columns=[self.target_col])
        y_train = df_train[self.target_col]

        rf = RandomForestRegressor(n_estimators=100, max_depth=8,
                                   min_samples_leaf=5, random_state=42, n_jobs=-1)
        rf.fit(X_train, y_train)

        importances = pd.DataFrame({
            'feature': X_train.columns,
            'importance': rf.feature_importances_
        }).sort_values('importance', ascending=False)

        top_features = importances.head(n_vars)['feature'].tolist()
        return (df_train[top_features + [self.target_col]],
                df_test[top_features  + [self.target_col]],
                top_features)

    # ------------------------------------------------------------------
    def apply_scaling(self, X_train, X_test, scaler_type='robust'):
        if scaler_type == 'robust':
            scaler = RobustScaler()
        elif scaler_type == 'standard':
            scaler = StandardScaler()
        elif scaler_type == 'power':
            scaler = PowerTransformer(method='yeo-johnson', standardize=True)
        else:
            return X_train, X_test
        return scaler.fit_transform(X_train), scaler.transform(X_test)

    # ------------------------------------------------------------------
    def evaluate_with_cv(self, X, y, model, cv=5):
        scores = cross_val_score(model, X, y, cv=cv, scoring='r2', n_jobs=-1)
        return scores.mean(), scores.std()

    # ------------------------------------------------------------------
    def train_and_evaluate(self, df_train, df_test, variant_name,
                           n_features, model_config, scaler_type='robust'):
        X_train = df_train.drop(columns=[self.target_col])
        y_train = df_train[self.target_col]
        X_test  = df_test.drop(columns=[self.target_col])
        y_test  = df_test[self.target_col]

        X_train_s, X_test_s = self.apply_scaling(X_train, X_test, scaler_type)

        mtype  = model_config['type']
        params = model_config['params']

        if mtype == 'gbm':
            model       = GradientBoostingRegressor(**params, random_state=42)
            params_short = (f"{params['n_estimators']}e_d{params['max_depth']}"
                            f"_lr{params['learning_rate']}")
        elif mtype == 'svm':
            model       = SVR(**params)
            params_short = f"SVM_{params.get('kernel','rbf')}_C{params.get('C',1)}"
        elif mtype == 'linear':
            model       = LinearRegression(**params, n_jobs=-1)
            params_short = "Linear_OLS"
        else:   # rf
            model       = RandomForestRegressor(**params, random_state=42, n_jobs=-1)
            params_short = (f"{params['n_estimators']}e_d{params['max_depth']}"
                            f"_msl{params['min_samples_leaf']}")

        cv_mean, cv_std = self.evaluate_with_cv(X_train_s, y_train, model)
        model.fit(X_train_s, y_train)

        y_pred_train = model.predict(X_train_s)
        y_pred_test  = model.predict(X_test_s)

        r2_train = r2_score(y_train, y_pred_train)
        r2_test  = r2_score(y_test,  y_pred_test)

        result = {
            'variant':      variant_name,
            'n_features':   n_features,
            'model':        mtype,
            'scaler':       scaler_type,
            'params_short': params_short,
            'r2_train':     r2_train,
            'r2_test':      r2_test,
            'r2_cv':        cv_mean,
            'cv_std':       cv_std,
            'rmse_test':    np.sqrt(mean_squared_error(y_test, y_pred_test)),
            'mae_test':     mean_absolute_error(y_test, y_pred_test),
            'overfit':      r2_train - r2_test,
        }
        self.results.append(result)
        return result

    # ------------------------------------------------------------------
    def _drop_leakage_cols(self, df):
        """
        Mantém como preditores APENAS:
          - area_plantada_ha
          - variáveis climáticas  (<var>_dec<N>_ano<M>)
          - ano  (contexto temporal)
        Tudo o mais é descartado (leakage ou identificadores).
        """
        # Colunas climáticas são preservadas automaticamente pela lógica de agregação;
        # aqui garantimos que nenhuma coluna não-climática não autorizada entre.
        keep = {self.target_col, 'area_plantada_ha'}

        drop_cols = [
            col for col in df.columns
            if col not in keep
            and not (col.startswith(tuple(f"{v}_dec" for v in self.climate_vars)))
        ]
        return df.drop(columns=drop_cols)

    # ------------------------------------------------------------------
    def run_experiments(self):
        print("=" * 120)
        print("DEFINIÇÃO DAS VARIANTES")
        print("=" * 120 + "\n")

        # Remover todas as colunas exceto target, area_plantada_ha, ano e climáticas
        # (identify_climate_variables já foi chamado em run() antes deste método)
        train_clean = self._drop_leakage_cols(self.train_df)
        test_clean  = self._drop_leakage_cols(self.test_df)

        # Normalizar dados brutos ANTES da agregação
        train_normalized, test_normalized = self.normalize_raw_data(train_clean, test_clean)

        variants = {
            'v1_original': {
                'name':        'V1: Original (3 dec early, 5-10 flow, 11-15 grain)',
                'early':       {'enabled': True,  'n_decendios': 3, 'stats': ['mean']},
                'flowering':   {'start_dec': 5,  'end_dec': 10, 'stats': ['mean']},
                'grain':       {'start_dec': 11, 'end_dec': 15, 'stats': ['mean']},
                'maturation':  {'enabled': False, 'start_dec': 16, 'end_dec': 18, 'stats': ['mean']},
            },
            'v6_with_variability': {
                'name':        'V6: Com variabilidade (mean + std)',
                'early':       {'enabled': True,  'n_decendios': 3, 'stats': ['mean', 'std']},
                'flowering':   {'start_dec': 5,  'end_dec': 10, 'stats': ['mean', 'std']},
                'grain':       {'start_dec': 11, 'end_dec': 15, 'stats': ['mean', 'std']},
                'maturation':  {'enabled': False, 'start_dec': 16, 'end_dec': 18, 'stats': ['mean']},
            },
            'v8_robust_stats': {
                'name':        'V8: Estatísticas robustas (mean + median)',
                'early':       {'enabled': True,  'n_decendios': 3, 'stats': ['mean']},
                'flowering':   {'start_dec': 5,  'end_dec': 10, 'stats': ['mean', 'median']},
                'grain':       {'start_dec': 11, 'end_dec': 15, 'stats': ['mean', 'median']},
                'maturation':  {'enabled': False, 'start_dec': 16, 'end_dec': 18, 'stats': ['mean']},
            },
            'v9_complete': {
                'name':        'V9: Completo (todas fases + variabilidade)',
                'early':       {'enabled': True,  'n_decendios': 4, 'stats': ['mean', 'std']},
                'flowering':   {'start_dec': 5,  'end_dec': 10, 'stats': ['mean', 'std']},
                'grain':       {'start_dec': 11, 'end_dec': 15, 'stats': ['mean', 'std']},
                'maturation':  {'enabled': True,  'start_dec': 16, 'end_dec': 18, 'stats': ['mean', 'std']},
            },
        }

        for key, cfg in variants.items():
            print(f"📋 {key}: {cfg['name']}")
        print()

        feature_counts = [35, 40, 45, 50]
        scalers        = ['robust', 'standard', 'power']
        model_configs  = [
            {'type': 'rf',     'params': {'n_estimators': 200, 'max_depth': 8,  'min_samples_split': 10, 'min_samples_leaf': 5}},
            {'type': 'rf',     'params': {'n_estimators': 250, 'max_depth': 8,  'min_samples_split': 10, 'min_samples_leaf': 5}},
            {'type': 'rf',     'params': {'n_estimators': 200, 'max_depth': 10, 'min_samples_split': 12, 'min_samples_leaf': 6}},
            {'type': 'gbm',    'params': {'n_estimators': 150, 'max_depth': 5,  'learning_rate': 0.03, 'subsample': 0.85}},
            {'type': 'gbm',    'params': {'n_estimators': 200, 'max_depth': 5,  'learning_rate': 0.02, 'subsample': 0.85}},
            {'type': 'svm',    'params': {'kernel': 'rbf',    'C': 10.0, 'epsilon': 0.1}},
            {'type': 'svm',    'params': {'kernel': 'linear', 'C': 1.0}},
            {'type': 'linear', 'params': {}},
        ]

        print("=" * 120)
        print("EXECUTANDO EXPERIMENTOS COM NORMALIZAÇÃO")
        print("=" * 120 + "\n")

        exp_num  = 0
        best_r2  = -float('inf')

        for var_key, var_config in variants.items():
            print(f"🔬 {var_config['name']}")

            train_agg = self.aggregate_features(train_normalized, var_config)
            test_agg  = self.aggregate_features(test_normalized,  var_config)

            n_features_available = train_agg.shape[1] - 1
            print(f"   Features disponíveis: {n_features_available}")

            for n_vars in feature_counts:
                if n_vars > n_features_available:
                    continue

                df_train_exp, df_test_exp, _ = self.select_features_rf(train_agg, test_agg, n_vars)

                for scaler_type in scalers:
                    for model_config in model_configs:
                        exp_num += 1
                        result   = self.train_and_evaluate(
                            df_train_exp, df_test_exp,
                            var_key, n_vars, model_config, scaler_type
                        )

                        if result['r2_test'] > best_r2:
                            best_r2 = result['r2_test']
                            print(f"   ⭐ [{exp_num:4d}] {model_config['type']} | {scaler_type:8s} | n={n_vars} | "
                                  f"R²Test={result['r2_test']:.4f} | R²CV={result['r2_cv']:.4f} | "
                                  f"Overfit={result['overfit']:.3f}")
                        elif result['r2_test'] > 0.28:
                            print(f"   ✓ [{exp_num:4d}] {model_config['type']} | {scaler_type:8s} | "
                                  f"n={n_vars} | R²={result['r2_test']:.4f}")
            print()

    # ------------------------------------------------------------------
    def display_results(self):
        print("=" * 120)
        print("🏆 TOP 30 RESULTADOS")
        print("=" * 120 + "\n")

        results_df = pd.DataFrame(self.results).sort_values('r2_test', ascending=False)

        print(f"{'#':<4} {'Variante':<22} {'n':<5} {'Modelo':<6} {'Scaler':<10} {'Config':<25} "
              f"{'R²Test':<9} {'R²CV':<9} {'Overfit':<8}")
        print("-" * 130)

        for idx, (_, row) in enumerate(results_df.head(30).iterrows(), 1):
            print(f"{idx:<4} {row['variant']:<22} {row['n_features']:<5} {row['model']:<6} "
                  f"{row['scaler']:<10} {row['params_short']:<25} "
                  f"{row['r2_test']:<9.4f} {row['r2_cv']:<9.4f} {row['overfit']:<8.3f}")

        best = results_df.iloc[0]

        print("\n" + "=" * 120)
        print("🎯 MELHOR CONFIGURAÇÃO COM NORMALIZAÇÃO")
        print("=" * 120)
        print(f"\nVariante: {best['variant']}")
        print(f"Features: {best['n_features']}")
        print(f"Modelo:   {best['model']}")
        print(f"Scaler:   {best['scaler']}")
        print(f"Config:   {best['params_short']}")
        print(f"\n📊 PERFORMANCE:")
        print(f"   R² Teste:    {best['r2_test']:.4f} ⭐")
        print(f"   R² CV:       {best['r2_cv']:.4f} ± {best['cv_std']:.4f}")
        print(f"   R² Treino:   {best['r2_train']:.4f}")
        print(f"   RMSE:        {best['rmse_test']:.2f} ton")
        print(f"   MAE:         {best['mae_test']:.2f} ton")
        print(f"   Overfitting: {best['overfit']:.4f}")

        print(f"\n📊 PERFORMANCE POR TIPO DE NORMALIZAÇÃO:")
        print("-" * 80)
        scaler_summary = results_df.groupby('scaler')['r2_test'].agg(['mean', 'max', 'count']).round(4)
        scaler_summary.columns = ['R² Médio', 'R² Máximo', 'Testes']
        print(scaler_summary.sort_values('R² Máximo', ascending=False))

        print(f"\n📊 PERFORMANCE POR VARIANTE:")
        print("-" * 80)
        variant_summary = results_df.groupby('variant')['r2_test'].agg(['mean', 'max', 'count']).round(4)
        variant_summary.columns = ['R² Médio', 'R² Máximo', 'Testes']
        print(variant_summary.sort_values('R² Máximo', ascending=False))

        print(f"\n📈 COMPARAÇÃO:")
        print(f"   Baseline (sem normalização):  R² = 0.2949")
        print(f"   Com normalização:             R² = {best['r2_test']:.4f}")
        improvement = (best['r2_test'] / 0.2949 - 1) * 100
        sign = '+' if improvement >= 0 else ''
        print(f"   Variação:                     {sign}{improvement:.2f}%")

        return results_df

    # ------------------------------------------------------------------
    def run(self):
        self.load_and_clean_data()
        self.identify_climate_variables()
        self.run_experiments()
        return self.display_results()


# ===============================================================================
# EXECUÇÃO
# ===============================================================================
if __name__ == "__main__":
    tester = CriticalV1VariantsTest(
        parquet_path = PARQUET_PATH,
        target_col   = TARGET_COL,
        year_col     = YEAR_COL,
        train_years  = TRAIN_YEARS,
        test_years   = TEST_YEARS,
    )
    results = tester.run()

    print("\n" + "=" * 120)
    print("✅ TESTE COM NORMALIZAÇÃO MELHORADA CONCLUÍDO!")
    print("=" * 120)

TESTE DE VARIAÇÕES DO CRITICAL_V1 - COM NORMALIZAÇÃO OTIMIZADA

✓ Dataset carregado: (2793, 8226)  |  colunas: ['cod_ibge', 'municipio', 'ano', 'cod_meso', 'mesorregiao', 'latitude', 'longitude', 'area_plantada_ha'] …

✓ Treino (2018–2022): (1995, 8226) | Teste (2023–2024): (798, 8226)

📊 TARGET (quantidade_produzida_ton) — ESTATÍSTICAS ORIGINAIS:
   Treino: média=44571.50, std=52797.57, min=0.00, max=423599.00
   Teste : média=50429.74, std=56091.13, min=0.00, max=416400.00

✓ 114 variáveis climáticas identificadas: ['AIRMASS', 'ALLSKY_KT', 'ALLSKY_NKT', 'ALLSKY_SFC_LW_DWN', 'ALLSKY_SFC_LW_UP'] …

DEFINIÇÃO DAS VARIANTES

🔧 Aplicando normalização nos dados brutos...
   ✓ 8208 colunas climáticas normalizadas
📋 v1_original: V1: Original (3 dec early, 5-10 flow, 11-15 grain)
📋 v6_with_variability: V6: Com variabilidade (mean + std)
📋 v8_robust_stats: V8: Estatísticas robustas (mean + median)
📋 v9_complete: V9: Completo (todas fases + variabilidade)

EXECUTANDO EXPERIMENTOS COM NORMALIZAÇ

## area_plantada_ha + variáveis climáticas ---> quantidade_produzida_ton

In [11]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import RobustScaler, StandardScaler, PowerTransformer
from sklearn.model_selection import cross_val_score
import warnings
warnings.filterwarnings('ignore')

# ===============================================================================
# CONFIGURAÇÃO - ajuste apenas estes caminhos se necessário
# ===============================================================================
PARQUET_PATH  = r'C:\Users\bruno\Desktop\Pipeline_TCC\data\processed\dataset_final.parquet'
TARGET_COL    = 'quantidade_produzida_ton'
YEAR_COL      = 'ano'
TRAIN_YEARS   = list(range(2018, 2023))   # 2018 a 2022 inclusive
TEST_YEARS    = [2023, 2024]
# ===============================================================================


class CriticalV1VariantsTest:
    """
    Teste focado em variações do critical_v1 com normalização otimizada.
    Lê um único arquivo parquet e faz o split por ano.
    """

    def __init__(self, parquet_path, target_col, year_col,
                 train_years, test_years):
        self.parquet_path = parquet_path
        self.target_col   = target_col
        self.year_col     = year_col
        self.train_years  = train_years
        self.test_years   = test_years
        self.results      = []

    # ------------------------------------------------------------------
    def load_and_clean_data(self):
        print("=" * 120)
        print("TESTE DE VARIAÇÕES DO CRITICAL_V1 - COM NORMALIZAÇÃO OTIMIZADA")
        print("=" * 120)

        df = pd.read_parquet(self.parquet_path)
        print(f"\n✓ Dataset carregado: {df.shape}  |  colunas: {df.columns.tolist()[:8]} …")

        # Split temporal
        self.train_df = df[df[self.year_col].isin(self.train_years)].dropna().reset_index(drop=True)
        self.test_df  = df[df[self.year_col].isin(self.test_years)].dropna().reset_index(drop=True)

        print(f"\n✓ Treino ({self.train_years[0]}–{self.train_years[-1]}): {self.train_df.shape} "
              f"| Teste ({self.test_years[0]}–{self.test_years[-1]}): {self.test_df.shape}")

        print(f"\n📊 TARGET ({self.target_col}) — ESTATÍSTICAS ORIGINAIS:")
        for label, subset in [("Treino", self.train_df), ("Teste ", self.test_df)]:
            s = subset[self.target_col]
            print(f"   {label}: média={s.mean():.2f}, std={s.std():.2f}, "
                  f"min={s.min():.2f}, max={s.max():.2f}")
        print()

    # ------------------------------------------------------------------
    def identify_climate_variables(self):
        """Detecta prefixos de variáveis climáticas do tipo <var>_dec<N>_ano<M>."""
        all_cols = self.train_df.columns.tolist()
        climate_vars = set()
        for col in all_cols:
            if 'dec' in col and 'ano' in col:
                var_name = col.split('dec')[0].rstrip('_')
                climate_vars.add(var_name)

        self.climate_vars = sorted(list(climate_vars))
        print(f"✓ {len(self.climate_vars)} variáveis climáticas identificadas: {self.climate_vars[:5]} …\n")

    # ------------------------------------------------------------------
    def get_variable_columns(self, var_name):
        return [col for col in self.train_df.columns
                if col.startswith(f"{var_name}_dec")]

    # ------------------------------------------------------------------
    def normalize_raw_data(self, train_df, test_df):
        """Normaliza colunas climáticas brutas ANTES da agregação."""
        print("🔧 Aplicando normalização nos dados brutos...")

        train_normalized = train_df.copy()
        test_normalized  = test_df.copy()

        climate_cols = [col for col in train_df.columns if 'dec' in col and 'ano' in col]

        for var in self.climate_vars:
            var_cols = [col for col in climate_cols if col.startswith(f"{var}_dec")]
            if var_cols:
                scaler = RobustScaler()
                train_normalized[var_cols] = scaler.fit_transform(train_df[var_cols])
                test_normalized[var_cols]  = scaler.transform(test_df[var_cols])

        print(f"   ✓ {len(climate_cols)} colunas climáticas normalizadas")
        return train_normalized, test_normalized

    # ------------------------------------------------------------------
    def aggregate_features(self, df, variant_config):
        """Agrega features por fase fenológica conforme configuração da variante."""
        non_climate_cols = [col for col in df.columns
                            if not any(col.startswith(f"{var}_dec")
                                       for var in self.climate_vars)]
        df_result = df[non_climate_cols].copy()

        for var in self.climate_vars:
            var_cols  = self.get_variable_columns(var)
            if not var_cols:
                continue

            ano1_cols = [c for c in var_cols if 'ano1' in c]
            ano2_cols = [c for c in var_cols if 'ano2' in c]

            # FASE 1: Pré-plantio (ano anterior)
            if variant_config['early']['enabled']:
                n = variant_config['early']['n_decendios']
                if ano1_cols and len(ano1_cols) >= n:
                    early_data = df[ano1_cols[-n:]]
                    cfg = variant_config['early']['stats']
                    if 'mean'   in cfg: df_result[f'{var}_early_mean']   = early_data.mean(axis=1)
                    if 'std'    in cfg: df_result[f'{var}_early_std']    = early_data.std(axis=1)
                    if 'min'    in cfg: df_result[f'{var}_early_min']    = early_data.min(axis=1)
                    if 'max'    in cfg: df_result[f'{var}_early_max']    = early_data.max(axis=1)

            # FASE 2: Florescimento
            fs, fe = variant_config['flowering']['start_dec'], variant_config['flowering']['end_dec']
            flowering_cols = [c for c in ano2_cols
                              if any(f'dec{d}_' in c for d in range(fs, fe + 1))]
            if flowering_cols:
                fd  = df[flowering_cols]
                cfg = variant_config['flowering']['stats']
                if 'mean'   in cfg: df_result[f'{var}_flowering_mean']   = fd.mean(axis=1)
                if 'std'    in cfg: df_result[f'{var}_flowering_std']    = fd.std(axis=1)
                if 'min'    in cfg: df_result[f'{var}_flowering_min']    = fd.min(axis=1)
                if 'max'    in cfg: df_result[f'{var}_flowering_max']    = fd.max(axis=1)
                if 'median' in cfg: df_result[f'{var}_flowering_median'] = fd.median(axis=1)

            # FASE 3: Enchimento de grãos
            gs, ge = variant_config['grain']['start_dec'], variant_config['grain']['end_dec']
            grain_cols = [c for c in ano2_cols
                          if any(f'dec{d}_' in c for d in range(gs, ge + 1))]
            if grain_cols:
                gd  = df[grain_cols]
                cfg = variant_config['grain']['stats']
                if 'mean'   in cfg: df_result[f'{var}_grain_mean']   = gd.mean(axis=1)
                if 'std'    in cfg: df_result[f'{var}_grain_std']    = gd.std(axis=1)
                if 'min'    in cfg: df_result[f'{var}_grain_min']    = gd.min(axis=1)
                if 'max'    in cfg: df_result[f'{var}_grain_max']    = gd.max(axis=1)
                if 'median' in cfg: df_result[f'{var}_grain_median'] = gd.median(axis=1)

            # FASE 4: Maturação (opcional)
            if variant_config['maturation']['enabled']:
                ms, me = variant_config['maturation']['start_dec'], variant_config['maturation']['end_dec']
                mat_cols = [c for c in ano2_cols
                            if any(f'dec{d}_' in c for d in range(ms, me + 1))]
                if mat_cols:
                    md  = df[mat_cols]
                    cfg = variant_config['maturation']['stats']
                    if 'mean' in cfg: df_result[f'{var}_maturation_mean'] = md.mean(axis=1)
                    if 'std'  in cfg: df_result[f'{var}_maturation_std']  = md.std(axis=1)

        return df_result

    # ------------------------------------------------------------------
    def select_features_rf(self, df_train, df_test, n_vars):
        X_train = df_train.drop(columns=[self.target_col])
        y_train = df_train[self.target_col]

        rf = RandomForestRegressor(n_estimators=100, max_depth=8,
                                   min_samples_leaf=5, random_state=42, n_jobs=-1)
        rf.fit(X_train, y_train)

        importances = pd.DataFrame({
            'feature': X_train.columns,
            'importance': rf.feature_importances_
        }).sort_values('importance', ascending=False)

        top_features = importances.head(n_vars)['feature'].tolist()
        return (df_train[top_features + [self.target_col]],
                df_test[top_features  + [self.target_col]],
                top_features)

    # ------------------------------------------------------------------
    def apply_scaling(self, X_train, X_test, scaler_type='robust'):
        if scaler_type == 'robust':
            scaler = RobustScaler()
        elif scaler_type == 'standard':
            scaler = StandardScaler()
        elif scaler_type == 'power':
            scaler = PowerTransformer(method='yeo-johnson', standardize=True)
        else:
            return X_train, X_test
        return scaler.fit_transform(X_train), scaler.transform(X_test)

    # ------------------------------------------------------------------
    def evaluate_with_cv(self, X, y, model, cv=5):
        scores = cross_val_score(model, X, y, cv=cv, scoring='r2', n_jobs=-1)
        return scores.mean(), scores.std()

    # ------------------------------------------------------------------
    def train_and_evaluate(self, df_train, df_test, variant_name,
                           n_features, model_config, scaler_type='robust'):
        X_train = df_train.drop(columns=[self.target_col])
        y_train = df_train[self.target_col]
        X_test  = df_test.drop(columns=[self.target_col])
        y_test  = df_test[self.target_col]

        X_train_s, X_test_s = self.apply_scaling(X_train, X_test, scaler_type)

        mtype  = model_config['type']
        params = model_config['params']

        if mtype == 'gbm':
            model       = GradientBoostingRegressor(**params, random_state=42)
            params_short = (f"{params['n_estimators']}e_d{params['max_depth']}"
                            f"_lr{params['learning_rate']}")
        elif mtype == 'svm':
            model       = SVR(**params)
            params_short = f"SVM_{params.get('kernel','rbf')}_C{params.get('C',1)}"
        elif mtype == 'linear':
            model       = LinearRegression(**params, n_jobs=-1)
            params_short = "Linear_OLS"
        else:   # rf
            model       = RandomForestRegressor(**params, random_state=42, n_jobs=-1)
            params_short = (f"{params['n_estimators']}e_d{params['max_depth']}"
                            f"_msl{params['min_samples_leaf']}")

        cv_mean, cv_std = self.evaluate_with_cv(X_train_s, y_train, model)
        model.fit(X_train_s, y_train)

        y_pred_train = model.predict(X_train_s)
        y_pred_test  = model.predict(X_test_s)

        r2_train = r2_score(y_train, y_pred_train)
        r2_test  = r2_score(y_test,  y_pred_test)

        result = {
            'variant':      variant_name,
            'n_features':   n_features,
            'model':        mtype,
            'scaler':       scaler_type,
            'params_short': params_short,
            'r2_train':     r2_train,
            'r2_test':      r2_test,
            'r2_cv':        cv_mean,
            'cv_std':       cv_std,
            'rmse_test':    np.sqrt(mean_squared_error(y_test, y_pred_test)),
            'mae_test':     mean_absolute_error(y_test, y_pred_test),
            'overfit':      r2_train - r2_test,
        }
        self.results.append(result)
        return result

    # ------------------------------------------------------------------
    def _drop_leakage_cols(self, df):
        """
        Mantém como preditores APENAS:
          - area_plantada_ha
          - variáveis climáticas  (<var>_dec<N>_ano<M>)
        'ano' é descartado para evitar vazamento intra-anual.
        Tudo o mais é descartado (leakage ou identificadores).
        """
        keep = {self.target_col, 'area_plantada_ha'}

        drop_cols = [
            col for col in df.columns
            if col not in keep
            and not (col.startswith(tuple(f"{v}_dec" for v in self.climate_vars)))
        ]
        return df.drop(columns=drop_cols)

    # ------------------------------------------------------------------
    def run_experiments(self):
        print("=" * 120)
        print("DEFINIÇÃO DAS VARIANTES")
        print("=" * 120 + "\n")

        # Manter apenas target, area_plantada_ha e climáticas (ano removido: sem vazamento intra-anual)
        # (identify_climate_variables já foi chamado em run() antes deste método)
        train_clean = self._drop_leakage_cols(self.train_df)
        test_clean  = self._drop_leakage_cols(self.test_df)

        # Normalizar dados brutos ANTES da agregação
        train_normalized, test_normalized = self.normalize_raw_data(train_clean, test_clean)

        variants = {
            'v1_original': {
                'name':        'V1: Original (3 dec early, 5-10 flow, 11-15 grain)',
                'early':       {'enabled': True,  'n_decendios': 3, 'stats': ['mean']},
                'flowering':   {'start_dec': 5,  'end_dec': 10, 'stats': ['mean']},
                'grain':       {'start_dec': 11, 'end_dec': 15, 'stats': ['mean']},
                'maturation':  {'enabled': False, 'start_dec': 16, 'end_dec': 18, 'stats': ['mean']},
            },
            'v6_with_variability': {
                'name':        'V6: Com variabilidade (mean + std)',
                'early':       {'enabled': True,  'n_decendios': 3, 'stats': ['mean', 'std']},
                'flowering':   {'start_dec': 5,  'end_dec': 10, 'stats': ['mean', 'std']},
                'grain':       {'start_dec': 11, 'end_dec': 15, 'stats': ['mean', 'std']},
                'maturation':  {'enabled': False, 'start_dec': 16, 'end_dec': 18, 'stats': ['mean']},
            },
            'v8_robust_stats': {
                'name':        'V8: Estatísticas robustas (mean + median)',
                'early':       {'enabled': True,  'n_decendios': 3, 'stats': ['mean']},
                'flowering':   {'start_dec': 5,  'end_dec': 10, 'stats': ['mean', 'median']},
                'grain':       {'start_dec': 11, 'end_dec': 15, 'stats': ['mean', 'median']},
                'maturation':  {'enabled': False, 'start_dec': 16, 'end_dec': 18, 'stats': ['mean']},
            },
            'v9_complete': {
                'name':        'V9: Completo (todas fases + variabilidade)',
                'early':       {'enabled': True,  'n_decendios': 4, 'stats': ['mean', 'std']},
                'flowering':   {'start_dec': 5,  'end_dec': 10, 'stats': ['mean', 'std']},
                'grain':       {'start_dec': 11, 'end_dec': 15, 'stats': ['mean', 'std']},
                'maturation':  {'enabled': True,  'start_dec': 16, 'end_dec': 18, 'stats': ['mean', 'std']},
            },
        }

        for key, cfg in variants.items():
            print(f"📋 {key}: {cfg['name']}")
        print()

        feature_counts = [35, 40, 45, 50]
        scalers        = ['robust', 'standard', 'power']
        model_configs  = [
            {'type': 'rf',     'params': {'n_estimators': 200, 'max_depth': 8,  'min_samples_split': 10, 'min_samples_leaf': 5}},
            {'type': 'rf',     'params': {'n_estimators': 250, 'max_depth': 8,  'min_samples_split': 10, 'min_samples_leaf': 5}},
            {'type': 'rf',     'params': {'n_estimators': 200, 'max_depth': 10, 'min_samples_split': 12, 'min_samples_leaf': 6}},
            {'type': 'gbm',    'params': {'n_estimators': 150, 'max_depth': 5,  'learning_rate': 0.03, 'subsample': 0.85}},
            {'type': 'gbm',    'params': {'n_estimators': 200, 'max_depth': 5,  'learning_rate': 0.02, 'subsample': 0.85}},
            {'type': 'svm',    'params': {'kernel': 'rbf',    'C': 10.0, 'epsilon': 0.1}},
            {'type': 'svm',    'params': {'kernel': 'linear', 'C': 1.0}},
            {'type': 'linear', 'params': {}},
        ]

        print("=" * 120)
        print("EXECUTANDO EXPERIMENTOS COM NORMALIZAÇÃO")
        print("=" * 120 + "\n")

        exp_num  = 0
        best_r2  = -float('inf')

        for var_key, var_config in variants.items():
            print(f"🔬 {var_config['name']}")

            train_agg = self.aggregate_features(train_normalized, var_config)
            test_agg  = self.aggregate_features(test_normalized,  var_config)

            n_features_available = train_agg.shape[1] - 1
            print(f"   Features disponíveis: {n_features_available}")

            for n_vars in feature_counts:
                if n_vars > n_features_available:
                    continue

                df_train_exp, df_test_exp, _ = self.select_features_rf(train_agg, test_agg, n_vars)

                for scaler_type in scalers:
                    for model_config in model_configs:
                        exp_num += 1
                        result   = self.train_and_evaluate(
                            df_train_exp, df_test_exp,
                            var_key, n_vars, model_config, scaler_type
                        )

                        if result['r2_test'] > best_r2:
                            best_r2 = result['r2_test']
                            print(f"   ⭐ [{exp_num:4d}] {model_config['type']} | {scaler_type:8s} | n={n_vars} | "
                                  f"R²Test={result['r2_test']:.4f} | R²CV={result['r2_cv']:.4f} | "
                                  f"Overfit={result['overfit']:.3f}")
                        elif result['r2_test'] > 0.28:
                            print(f"   ✓ [{exp_num:4d}] {model_config['type']} | {scaler_type:8s} | "
                                  f"n={n_vars} | R²={result['r2_test']:.4f}")
            print()

    # ------------------------------------------------------------------
    def display_results(self):
        print("=" * 120)
        print("🏆 TOP 30 RESULTADOS")
        print("=" * 120 + "\n")

        results_df = pd.DataFrame(self.results).sort_values('r2_test', ascending=False)

        print(f"{'#':<4} {'Variante':<22} {'n':<5} {'Modelo':<6} {'Scaler':<10} {'Config':<25} "
              f"{'R²Test':<9} {'R²CV':<9} {'Overfit':<8}")
        print("-" * 130)

        for idx, (_, row) in enumerate(results_df.head(30).iterrows(), 1):
            print(f"{idx:<4} {row['variant']:<22} {row['n_features']:<5} {row['model']:<6} "
                  f"{row['scaler']:<10} {row['params_short']:<25} "
                  f"{row['r2_test']:<9.4f} {row['r2_cv']:<9.4f} {row['overfit']:<8.3f}")

        best = results_df.iloc[0]

        print("\n" + "=" * 120)
        print("🎯 MELHOR CONFIGURAÇÃO COM NORMALIZAÇÃO")
        print("=" * 120)
        print(f"\nVariante: {best['variant']}")
        print(f"Features: {best['n_features']}")
        print(f"Modelo:   {best['model']}")
        print(f"Scaler:   {best['scaler']}")
        print(f"Config:   {best['params_short']}")
        print(f"\n📊 PERFORMANCE:")
        print(f"   R² Teste:    {best['r2_test']:.4f} ⭐")
        print(f"   R² CV:       {best['r2_cv']:.4f} ± {best['cv_std']:.4f}")
        print(f"   R² Treino:   {best['r2_train']:.4f}")
        print(f"   RMSE:        {best['rmse_test']:.2f} ton")
        print(f"   MAE:         {best['mae_test']:.2f} ton")
        print(f"   Overfitting: {best['overfit']:.4f}")

        print(f"\n📊 PERFORMANCE POR TIPO DE NORMALIZAÇÃO:")
        print("-" * 80)
        scaler_summary = results_df.groupby('scaler')['r2_test'].agg(['mean', 'max', 'count']).round(4)
        scaler_summary.columns = ['R² Médio', 'R² Máximo', 'Testes']
        print(scaler_summary.sort_values('R² Máximo', ascending=False))

        print(f"\n📊 PERFORMANCE POR VARIANTE:")
        print("-" * 80)
        variant_summary = results_df.groupby('variant')['r2_test'].agg(['mean', 'max', 'count']).round(4)
        variant_summary.columns = ['R² Médio', 'R² Máximo', 'Testes']
        print(variant_summary.sort_values('R² Máximo', ascending=False))

        print(f"\n📈 COMPARAÇÃO:")
        print(f"   Baseline (sem normalização):  R² = 0.2949")
        print(f"   Com normalização:             R² = {best['r2_test']:.4f}")
        improvement = (best['r2_test'] / 0.2949 - 1) * 100
        sign = '+' if improvement >= 0 else ''
        print(f"   Variação:                     {sign}{improvement:.2f}%")

        return results_df

    # ------------------------------------------------------------------
    def run(self):
        self.load_and_clean_data()
        self.identify_climate_variables()
        self.run_experiments()
        return self.display_results()


# ===============================================================================
# EXECUÇÃO
# ===============================================================================
if __name__ == "__main__":
    tester = CriticalV1VariantsTest(
        parquet_path = PARQUET_PATH,
        target_col   = TARGET_COL,
        year_col     = YEAR_COL,
        train_years  = TRAIN_YEARS,
        test_years   = TEST_YEARS,
    )
    results = tester.run()

    print("\n" + "=" * 120)
    print("✅ TESTE COM NORMALIZAÇÃO MELHORADA CONCLUÍDO!")
    print("=" * 120)

TESTE DE VARIAÇÕES DO CRITICAL_V1 - COM NORMALIZAÇÃO OTIMIZADA

✓ Dataset carregado: (2793, 8226)  |  colunas: ['cod_ibge', 'municipio', 'ano', 'cod_meso', 'mesorregiao', 'latitude', 'longitude', 'area_plantada_ha'] …

✓ Treino (2018–2022): (1995, 8226) | Teste (2023–2024): (798, 8226)

📊 TARGET (quantidade_produzida_ton) — ESTATÍSTICAS ORIGINAIS:
   Treino: média=44571.50, std=52797.57, min=0.00, max=423599.00
   Teste : média=50429.74, std=56091.13, min=0.00, max=416400.00

✓ 114 variáveis climáticas identificadas: ['AIRMASS', 'ALLSKY_KT', 'ALLSKY_NKT', 'ALLSKY_SFC_LW_DWN', 'ALLSKY_SFC_LW_UP'] …

DEFINIÇÃO DAS VARIANTES

🔧 Aplicando normalização nos dados brutos...
   ✓ 8208 colunas climáticas normalizadas
📋 v1_original: V1: Original (3 dec early, 5-10 flow, 11-15 grain)
📋 v6_with_variability: V6: Com variabilidade (mean + std)
📋 v8_robust_stats: V8: Estatísticas robustas (mean + median)
📋 v9_complete: V9: Completo (todas fases + variabilidade)

EXECUTANDO EXPERIMENTOS COM NORMALIZAÇ

## variáveis climáticas ---> valor_por_ha' = 'valor_producao_mil_reais' * 1000 / 'area_plantada_ha'

In [7]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import RobustScaler, StandardScaler, PowerTransformer
from sklearn.model_selection import cross_val_score
import warnings
warnings.filterwarnings('ignore')

# ===============================================================================
# CONFIGURAÇÃO - ajuste apenas estes caminhos se necessário
# ===============================================================================
PARQUET_PATH  = r'C:\Users\bruno\Desktop\Pipeline_TCC\data\processed\dataset_final.parquet'
TARGET_COL    = 'valor_por_ha'          # calculado em load_and_clean_data
YEAR_COL      = 'ano'
TRAIN_YEARS   = list(range(2018, 2023))   # 2018 a 2022 inclusive
TEST_YEARS    = [2023, 2024]
# ===============================================================================


class CriticalV1VariantsTest:
    """
    Teste focado em variações do critical_v1 com normalização otimizada.
    Lê um único arquivo parquet e faz o split por ano.
    """

    def __init__(self, parquet_path, target_col, year_col,
                 train_years, test_years):
        self.parquet_path = parquet_path
        self.target_col   = target_col
        self.year_col     = year_col
        self.train_years  = train_years
        self.test_years   = test_years
        self.results      = []

    # ------------------------------------------------------------------
    def load_and_clean_data(self):
        print("=" * 120)
        print("TESTE DE VARIAÇÕES DO CRITICAL_V1 - COM NORMALIZAÇÃO OTIMIZADA")
        print("=" * 120)

        df = pd.read_parquet(self.parquet_path)
        print(f"\n✓ Dataset carregado: {df.shape}  |  colunas: {df.columns.tolist()[:8]} …")

        # ── Target derivado: valor_producao_mil_reais * 1000 / area_plantada_ha ──
        df[self.target_col] = (df['valor_producao_mil_reais'] * 1000) / df['area_plantada_ha']

        # Split temporal
        self.train_df = df[df[self.year_col].isin(self.train_years)].dropna().reset_index(drop=True)
        self.test_df  = df[df[self.year_col].isin(self.test_years)].dropna().reset_index(drop=True)

        print(f"\n✓ Treino ({self.train_years[0]}–{self.train_years[-1]}): {self.train_df.shape} "
              f"| Teste ({self.test_years[0]}–{self.test_years[-1]}): {self.test_df.shape}")

        print(f"\n📊 TARGET ({self.target_col}) — ESTATÍSTICAS ORIGINAIS:")
        for label, subset in [("Treino", self.train_df), ("Teste ", self.test_df)]:
            s = subset[self.target_col]
            print(f"   {label}: média={s.mean():.2f}, std={s.std():.2f}, "
                  f"min={s.min():.2f}, max={s.max():.2f}")
        print()

    # ------------------------------------------------------------------
    def identify_climate_variables(self):
        """Detecta prefixos de variáveis climáticas do tipo <var>_dec<N>_ano<M>."""
        all_cols = self.train_df.columns.tolist()
        climate_vars = set()
        for col in all_cols:
            if 'dec' in col and 'ano' in col:
                var_name = col.split('dec')[0].rstrip('_')
                climate_vars.add(var_name)

        self.climate_vars = sorted(list(climate_vars))
        print(f"✓ {len(self.climate_vars)} variáveis climáticas identificadas: {self.climate_vars[:5]} …\n")

    # ------------------------------------------------------------------
    def get_variable_columns(self, var_name):
        return [col for col in self.train_df.columns
                if col.startswith(f"{var_name}_dec")]

    # ------------------------------------------------------------------
    def normalize_raw_data(self, train_df, test_df):
        """Normaliza colunas climáticas brutas ANTES da agregação."""
        print("🔧 Aplicando normalização nos dados brutos...")

        train_normalized = train_df.copy()
        test_normalized  = test_df.copy()

        climate_cols = [col for col in train_df.columns if 'dec' in col and 'ano' in col]

        for var in self.climate_vars:
            var_cols = [col for col in climate_cols if col.startswith(f"{var}_dec")]
            if var_cols:
                scaler = RobustScaler()
                train_normalized[var_cols] = scaler.fit_transform(train_df[var_cols])
                test_normalized[var_cols]  = scaler.transform(test_df[var_cols])

        print(f"   ✓ {len(climate_cols)} colunas climáticas normalizadas")
        return train_normalized, test_normalized

    # ------------------------------------------------------------------
    def aggregate_features(self, df, variant_config):
        """Agrega features por fase fenológica conforme configuração da variante."""
        non_climate_cols = [col for col in df.columns
                            if not any(col.startswith(f"{var}_dec")
                                       for var in self.climate_vars)]
        df_result = df[non_climate_cols].copy()

        for var in self.climate_vars:
            var_cols  = self.get_variable_columns(var)
            if not var_cols:
                continue

            ano1_cols = [c for c in var_cols if 'ano1' in c]
            ano2_cols = [c for c in var_cols if 'ano2' in c]

            # FASE 1: Pré-plantio (ano anterior)
            if variant_config['early']['enabled']:
                n = variant_config['early']['n_decendios']
                if ano1_cols and len(ano1_cols) >= n:
                    early_data = df[ano1_cols[-n:]]
                    cfg = variant_config['early']['stats']
                    if 'mean'   in cfg: df_result[f'{var}_early_mean']   = early_data.mean(axis=1)
                    if 'std'    in cfg: df_result[f'{var}_early_std']    = early_data.std(axis=1)
                    if 'min'    in cfg: df_result[f'{var}_early_min']    = early_data.min(axis=1)
                    if 'max'    in cfg: df_result[f'{var}_early_max']    = early_data.max(axis=1)

            # FASE 2: Florescimento
            fs, fe = variant_config['flowering']['start_dec'], variant_config['flowering']['end_dec']
            flowering_cols = [c for c in ano2_cols
                              if any(f'dec{d}_' in c for d in range(fs, fe + 1))]
            if flowering_cols:
                fd  = df[flowering_cols]
                cfg = variant_config['flowering']['stats']
                if 'mean'   in cfg: df_result[f'{var}_flowering_mean']   = fd.mean(axis=1)
                if 'std'    in cfg: df_result[f'{var}_flowering_std']    = fd.std(axis=1)
                if 'min'    in cfg: df_result[f'{var}_flowering_min']    = fd.min(axis=1)
                if 'max'    in cfg: df_result[f'{var}_flowering_max']    = fd.max(axis=1)
                if 'median' in cfg: df_result[f'{var}_flowering_median'] = fd.median(axis=1)

            # FASE 3: Enchimento de grãos
            gs, ge = variant_config['grain']['start_dec'], variant_config['grain']['end_dec']
            grain_cols = [c for c in ano2_cols
                          if any(f'dec{d}_' in c for d in range(gs, ge + 1))]
            if grain_cols:
                gd  = df[grain_cols]
                cfg = variant_config['grain']['stats']
                if 'mean'   in cfg: df_result[f'{var}_grain_mean']   = gd.mean(axis=1)
                if 'std'    in cfg: df_result[f'{var}_grain_std']    = gd.std(axis=1)
                if 'min'    in cfg: df_result[f'{var}_grain_min']    = gd.min(axis=1)
                if 'max'    in cfg: df_result[f'{var}_grain_max']    = gd.max(axis=1)
                if 'median' in cfg: df_result[f'{var}_grain_median'] = gd.median(axis=1)

            # FASE 4: Maturação (opcional)
            if variant_config['maturation']['enabled']:
                ms, me = variant_config['maturation']['start_dec'], variant_config['maturation']['end_dec']
                mat_cols = [c for c in ano2_cols
                            if any(f'dec{d}_' in c for d in range(ms, me + 1))]
                if mat_cols:
                    md  = df[mat_cols]
                    cfg = variant_config['maturation']['stats']
                    if 'mean' in cfg: df_result[f'{var}_maturation_mean'] = md.mean(axis=1)
                    if 'std'  in cfg: df_result[f'{var}_maturation_std']  = md.std(axis=1)

        return df_result

    # ------------------------------------------------------------------
    def select_features_rf(self, df_train, df_test, n_vars):
        X_train = df_train.drop(columns=[self.target_col])
        y_train = df_train[self.target_col]

        rf = RandomForestRegressor(n_estimators=100, max_depth=8,
                                   min_samples_leaf=5, random_state=42, n_jobs=-1)
        rf.fit(X_train, y_train)

        importances = pd.DataFrame({
            'feature': X_train.columns,
            'importance': rf.feature_importances_
        }).sort_values('importance', ascending=False)

        top_features = importances.head(n_vars)['feature'].tolist()
        return (df_train[top_features + [self.target_col]],
                df_test[top_features  + [self.target_col]],
                top_features)

    # ------------------------------------------------------------------
    def apply_scaling(self, X_train, X_test, scaler_type='robust'):
        if scaler_type == 'robust':
            scaler = RobustScaler()
        elif scaler_type == 'standard':
            scaler = StandardScaler()
        elif scaler_type == 'power':
            scaler = PowerTransformer(method='yeo-johnson', standardize=True)
        else:
            return X_train, X_test
        return scaler.fit_transform(X_train), scaler.transform(X_test)

    # ------------------------------------------------------------------
    def evaluate_with_cv(self, X, y, model, cv=5):
        scores = cross_val_score(model, X, y, cv=cv, scoring='r2', n_jobs=-1)
        return scores.mean(), scores.std()

    # ------------------------------------------------------------------
    def train_and_evaluate(self, df_train, df_test, variant_name,
                           n_features, model_config, scaler_type='robust'):
        X_train = df_train.drop(columns=[self.target_col])
        y_train = df_train[self.target_col]
        X_test  = df_test.drop(columns=[self.target_col])
        y_test  = df_test[self.target_col]

        X_train_s, X_test_s = self.apply_scaling(X_train, X_test, scaler_type)

        mtype  = model_config['type']
        params = model_config['params']

        if mtype == 'gbm':
            model       = GradientBoostingRegressor(**params, random_state=42)
            params_short = (f"{params['n_estimators']}e_d{params['max_depth']}"
                            f"_lr{params['learning_rate']}")
        elif mtype == 'svm':
            model       = SVR(**params)
            params_short = f"SVM_{params.get('kernel','rbf')}_C{params.get('C',1)}"
        elif mtype == 'linear':
            model       = LinearRegression(**params, n_jobs=-1)
            params_short = "Linear_OLS"
        else:   # rf
            model       = RandomForestRegressor(**params, random_state=42, n_jobs=-1)
            params_short = (f"{params['n_estimators']}e_d{params['max_depth']}"
                            f"_msl{params['min_samples_leaf']}")

        cv_mean, cv_std = self.evaluate_with_cv(X_train_s, y_train, model)
        model.fit(X_train_s, y_train)

        y_pred_train = model.predict(X_train_s)
        y_pred_test  = model.predict(X_test_s)

        r2_train = r2_score(y_train, y_pred_train)
        r2_test  = r2_score(y_test,  y_pred_test)

        result = {
            'variant':      variant_name,
            'n_features':   n_features,
            'model':        mtype,
            'scaler':       scaler_type,
            'params_short': params_short,
            'r2_train':     r2_train,
            'r2_test':      r2_test,
            'r2_cv':        cv_mean,
            'cv_std':       cv_std,
            'rmse_test':    np.sqrt(mean_squared_error(y_test, y_pred_test)),
            'mae_test':     mean_absolute_error(y_test, y_pred_test),
            'overfit':      r2_train - r2_test,
        }
        self.results.append(result)
        return result

    # ------------------------------------------------------------------
    def _drop_leakage_cols(self, df):
        """
        Mantém APENAS o target e as variáveis climáticas brutas (<var>_dec<N>_ano<M>).
        Tudo o mais (identificadores, produção, área, valor, IPCA…) é descartado.
        """
        # Manter APENAS o target e as colunas climáticas brutas (<var>_dec<N>_ano<M>)
        climate_prefixes = tuple(f"{v}_dec" for v in self.climate_vars)
        keep_cols = [
            col for col in df.columns
            if col == self.target_col or col.startswith(climate_prefixes)
        ]
        return df[keep_cols]

    # ------------------------------------------------------------------
    def run_experiments(self):
        print("=" * 120)
        print("DEFINIÇÃO DAS VARIANTES")
        print("=" * 120 + "\n")

        # Manter apenas target e variáveis climáticas (tudo mais descartado)
        train_clean = self._drop_leakage_cols(self.train_df)
        test_clean  = self._drop_leakage_cols(self.test_df)

        # Normalizar dados brutos ANTES da agregação
        train_normalized, test_normalized = self.normalize_raw_data(train_clean, test_clean)

        variants = {
            'v1_original': {
                'name':        'V1: Original (3 dec early, 5-10 flow, 11-15 grain)',
                'early':       {'enabled': True,  'n_decendios': 3, 'stats': ['mean']},
                'flowering':   {'start_dec': 5,  'end_dec': 10, 'stats': ['mean']},
                'grain':       {'start_dec': 11, 'end_dec': 15, 'stats': ['mean']},
                'maturation':  {'enabled': False, 'start_dec': 16, 'end_dec': 18, 'stats': ['mean']},
            },
            'v6_with_variability': {
                'name':        'V6: Com variabilidade (mean + std)',
                'early':       {'enabled': True,  'n_decendios': 3, 'stats': ['mean', 'std']},
                'flowering':   {'start_dec': 5,  'end_dec': 10, 'stats': ['mean', 'std']},
                'grain':       {'start_dec': 11, 'end_dec': 15, 'stats': ['mean', 'std']},
                'maturation':  {'enabled': False, 'start_dec': 16, 'end_dec': 18, 'stats': ['mean']},
            },
            'v8_robust_stats': {
                'name':        'V8: Estatísticas robustas (mean + median)',
                'early':       {'enabled': True,  'n_decendios': 3, 'stats': ['mean']},
                'flowering':   {'start_dec': 5,  'end_dec': 10, 'stats': ['mean', 'median']},
                'grain':       {'start_dec': 11, 'end_dec': 15, 'stats': ['mean', 'median']},
                'maturation':  {'enabled': False, 'start_dec': 16, 'end_dec': 18, 'stats': ['mean']},
            },
            'v9_complete': {
                'name':        'V9: Completo (todas fases + variabilidade)',
                'early':       {'enabled': True,  'n_decendios': 4, 'stats': ['mean', 'std']},
                'flowering':   {'start_dec': 5,  'end_dec': 10, 'stats': ['mean', 'std']},
                'grain':       {'start_dec': 11, 'end_dec': 15, 'stats': ['mean', 'std']},
                'maturation':  {'enabled': True,  'start_dec': 16, 'end_dec': 18, 'stats': ['mean', 'std']},
            },
        }

        for key, cfg in variants.items():
            print(f"📋 {key}: {cfg['name']}")
        print()

        feature_counts = [35, 40, 45, 50]
        scalers        = ['robust', 'standard', 'power']
        model_configs  = [
            {'type': 'rf',     'params': {'n_estimators': 200, 'max_depth': 8,  'min_samples_split': 10, 'min_samples_leaf': 5}},
            {'type': 'rf',     'params': {'n_estimators': 250, 'max_depth': 8,  'min_samples_split': 10, 'min_samples_leaf': 5}},
            {'type': 'rf',     'params': {'n_estimators': 200, 'max_depth': 10, 'min_samples_split': 12, 'min_samples_leaf': 6}},
            {'type': 'gbm',    'params': {'n_estimators': 150, 'max_depth': 5,  'learning_rate': 0.03, 'subsample': 0.85}},
            {'type': 'gbm',    'params': {'n_estimators': 200, 'max_depth': 5,  'learning_rate': 0.02, 'subsample': 0.85}},
            {'type': 'svm',    'params': {'kernel': 'rbf',    'C': 10.0, 'epsilon': 0.1}},
            {'type': 'svm',    'params': {'kernel': 'linear', 'C': 1.0}},
            {'type': 'linear', 'params': {}},
        ]

        print("=" * 120)
        print("EXECUTANDO EXPERIMENTOS COM NORMALIZAÇÃO")
        print("=" * 120 + "\n")

        exp_num  = 0
        best_r2  = -float('inf')

        for var_key, var_config in variants.items():
            print(f"🔬 {var_config['name']}")

            train_agg = self.aggregate_features(train_normalized, var_config)
            test_agg  = self.aggregate_features(test_normalized,  var_config)

            n_features_available = train_agg.shape[1] - 1
            print(f"   Features disponíveis: {n_features_available}")

            for n_vars in feature_counts:
                if n_vars > n_features_available:
                    continue

                df_train_exp, df_test_exp, _ = self.select_features_rf(train_agg, test_agg, n_vars)

                for scaler_type in scalers:
                    for model_config in model_configs:
                        exp_num += 1
                        result   = self.train_and_evaluate(
                            df_train_exp, df_test_exp,
                            var_key, n_vars, model_config, scaler_type
                        )

                        if result['r2_test'] > best_r2:
                            best_r2 = result['r2_test']
                            print(f"   ⭐ [{exp_num:4d}] {model_config['type']} | {scaler_type:8s} | n={n_vars} | "
                                  f"R²Test={result['r2_test']:.4f} | R²CV={result['r2_cv']:.4f} | "
                                  f"Overfit={result['overfit']:.3f}")
                        elif result['r2_test'] > 0.28:
                            print(f"   ✓ [{exp_num:4d}] {model_config['type']} | {scaler_type:8s} | "
                                  f"n={n_vars} | R²={result['r2_test']:.4f}")
            print()

    # ------------------------------------------------------------------
    def display_results(self):
        print("=" * 120)
        print("🏆 TOP 30 RESULTADOS")
        print("=" * 120 + "\n")

        results_df = pd.DataFrame(self.results).sort_values('r2_test', ascending=False)

        print(f"{'#':<4} {'Variante':<22} {'n':<5} {'Modelo':<6} {'Scaler':<10} {'Config':<25} "
              f"{'R²Test':<9} {'R²CV':<9} {'Overfit':<8}")
        print("-" * 130)

        for idx, (_, row) in enumerate(results_df.head(30).iterrows(), 1):
            print(f"{idx:<4} {row['variant']:<22} {row['n_features']:<5} {row['model']:<6} "
                  f"{row['scaler']:<10} {row['params_short']:<25} "
                  f"{row['r2_test']:<9.4f} {row['r2_cv']:<9.4f} {row['overfit']:<8.3f}")

        best = results_df.iloc[0]

        print("\n" + "=" * 120)
        print("🎯 MELHOR CONFIGURAÇÃO COM NORMALIZAÇÃO")
        print("=" * 120)
        print(f"\nVariante: {best['variant']}")
        print(f"Features: {best['n_features']}")
        print(f"Modelo:   {best['model']}")
        print(f"Scaler:   {best['scaler']}")
        print(f"Config:   {best['params_short']}")
        print(f"\n📊 PERFORMANCE:")
        print(f"   R² Teste:    {best['r2_test']:.4f} ⭐")
        print(f"   R² CV:       {best['r2_cv']:.4f} ± {best['cv_std']:.4f}")
        print(f"   R² Treino:   {best['r2_train']:.4f}")
        print(f"   RMSE:        {best['rmse_test']:.2f} R$/ha")
        print(f"   MAE:         {best['mae_test']:.2f} R$/ha")
        print(f"   Overfitting: {best['overfit']:.4f}")

        print(f"\n📊 PERFORMANCE POR TIPO DE NORMALIZAÇÃO:")
        print("-" * 80)
        scaler_summary = results_df.groupby('scaler')['r2_test'].agg(['mean', 'max', 'count']).round(4)
        scaler_summary.columns = ['R² Médio', 'R² Máximo', 'Testes']
        print(scaler_summary.sort_values('R² Máximo', ascending=False))

        print(f"\n📊 PERFORMANCE POR VARIANTE:")
        print("-" * 80)
        variant_summary = results_df.groupby('variant')['r2_test'].agg(['mean', 'max', 'count']).round(4)
        variant_summary.columns = ['R² Médio', 'R² Máximo', 'Testes']
        print(variant_summary.sort_values('R² Máximo', ascending=False))

        print(f"\n📈 COMPARAÇÃO:")
        print(f"   Baseline (sem normalização):  R² = 0.2949")
        print(f"   Com normalização:             R² = {best['r2_test']:.4f}")
        improvement = (best['r2_test'] / 0.2949 - 1) * 100
        sign = '+' if improvement >= 0 else ''
        print(f"   Variação:                     {sign}{improvement:.2f}%")

        return results_df

    # ------------------------------------------------------------------
    def run(self):
        self.load_and_clean_data()
        self.identify_climate_variables()
        self.run_experiments()
        return self.display_results()


# ===============================================================================
# EXECUÇÃO
# ===============================================================================
if __name__ == "__main__":
    tester = CriticalV1VariantsTest(
        parquet_path = PARQUET_PATH,
        target_col   = TARGET_COL,
        year_col     = YEAR_COL,
        train_years  = TRAIN_YEARS,
        test_years   = TEST_YEARS,
    )
    results = tester.run()

    print("\n" + "=" * 120)
    print("✅ TESTE COM NORMALIZAÇÃO MELHORADA CONCLUÍDO!")
    print("=" * 120)

TESTE DE VARIAÇÕES DO CRITICAL_V1 - COM NORMALIZAÇÃO OTIMIZADA

✓ Dataset carregado: (2793, 8226)  |  colunas: ['cod_ibge', 'municipio', 'ano', 'cod_meso', 'mesorregiao', 'latitude', 'longitude', 'area_plantada_ha'] …

✓ Treino (2018–2022): (1910, 8227) | Teste (2023–2024): (780, 8227)

📊 TARGET (valor_por_ha) — ESTATÍSTICAS ORIGINAIS:
   Treino: média=5480.42, std=2616.61, min=606.85, max=12600.00
   Teste : média=7196.69, std=2038.40, min=0.00, max=12656.76

✓ 114 variáveis climáticas identificadas: ['AIRMASS', 'ALLSKY_KT', 'ALLSKY_NKT', 'ALLSKY_SFC_LW_DWN', 'ALLSKY_SFC_LW_UP'] …

DEFINIÇÃO DAS VARIANTES

🔧 Aplicando normalização nos dados brutos...
   ✓ 8208 colunas climáticas normalizadas
📋 v1_original: V1: Original (3 dec early, 5-10 flow, 11-15 grain)
📋 v6_with_variability: V6: Com variabilidade (mean + std)
📋 v8_robust_stats: V8: Estatísticas robustas (mean + median)
📋 v9_complete: V9: Completo (todas fases + variabilidade)

EXECUTANDO EXPERIMENTOS COM NORMALIZAÇÃO

🔬 V1: Origin

## váriaveis climáticas --> rendimento_kg_ha

In [12]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import RobustScaler, StandardScaler, PowerTransformer
from sklearn.model_selection import cross_val_score
import warnings
warnings.filterwarnings('ignore')

# ===============================================================================
# CONFIGURAÇÃO - ajuste apenas estes caminhos se necessário
# ===============================================================================
PARQUET_PATH  = r'C:\Users\bruno\Desktop\Pipeline_TCC\data\processed\dataset_final.parquet'
TARGET_COL    = 'rendimento_kg_ha'
YEAR_COL      = 'ano'
TRAIN_YEARS   = list(range(2018, 2023))   # 2018 a 2022 inclusive
TEST_YEARS    = [2023, 2024]
# ===============================================================================


class CriticalV1VariantsTest:
    """
    Teste focado em variações do critical_v1 com normalização otimizada.
    Lê um único arquivo parquet e faz o split por ano.
    """

    def __init__(self, parquet_path, target_col, year_col,
                 train_years, test_years):
        self.parquet_path = parquet_path
        self.target_col   = target_col
        self.year_col     = year_col
        self.train_years  = train_years
        self.test_years   = test_years
        self.results      = []

    # ------------------------------------------------------------------
    def load_and_clean_data(self):
        print("=" * 120)
        print("TESTE DE VARIAÇÕES DO CRITICAL_V1 - COM NORMALIZAÇÃO OTIMIZADA")
        print("=" * 120)

        df = pd.read_parquet(self.parquet_path)
        print(f"\n✓ Dataset carregado: {df.shape}  |  colunas: {df.columns.tolist()[:8]} …")

        # Split temporal
        self.train_df = df[df[self.year_col].isin(self.train_years)].dropna().reset_index(drop=True)
        self.test_df  = df[df[self.year_col].isin(self.test_years)].dropna().reset_index(drop=True)

        print(f"\n✓ Treino ({self.train_years[0]}–{self.train_years[-1]}): {self.train_df.shape} "
              f"| Teste ({self.test_years[0]}–{self.test_years[-1]}): {self.test_df.shape}")

        print(f"\n📊 TARGET ({self.target_col}) — ESTATÍSTICAS ORIGINAIS:")
        for label, subset in [("Treino", self.train_df), ("Teste ", self.test_df)]:
            s = subset[self.target_col]
            print(f"   {label}: média={s.mean():.2f}, std={s.std():.2f}, "
                  f"min={s.min():.2f}, max={s.max():.2f}")
        print()

    # ------------------------------------------------------------------
    def identify_climate_variables(self):
        """Detecta prefixos de variáveis climáticas do tipo <var>_dec<N>_ano<M>."""
        all_cols = self.train_df.columns.tolist()
        climate_vars = set()
        for col in all_cols:
            if 'dec' in col and 'ano' in col:
                var_name = col.split('dec')[0].rstrip('_')
                climate_vars.add(var_name)

        self.climate_vars = sorted(list(climate_vars))
        print(f"✓ {len(self.climate_vars)} variáveis climáticas identificadas: {self.climate_vars[:5]} …\n")

    # ------------------------------------------------------------------
    def get_variable_columns(self, var_name):
        return [col for col in self.train_df.columns
                if col.startswith(f"{var_name}_dec")]

    # ------------------------------------------------------------------
    def normalize_raw_data(self, train_df, test_df):
        """Normaliza colunas climáticas brutas ANTES da agregação."""
        print("🔧 Aplicando normalização nos dados brutos...")

        train_normalized = train_df.copy()
        test_normalized  = test_df.copy()

        climate_cols = [col for col in train_df.columns if 'dec' in col and 'ano' in col]

        for var in self.climate_vars:
            var_cols = [col for col in climate_cols if col.startswith(f"{var}_dec")]
            if var_cols:
                scaler = RobustScaler()
                train_normalized[var_cols] = scaler.fit_transform(train_df[var_cols])
                test_normalized[var_cols]  = scaler.transform(test_df[var_cols])

        print(f"   ✓ {len(climate_cols)} colunas climáticas normalizadas")
        return train_normalized, test_normalized

    # ------------------------------------------------------------------
    def aggregate_features(self, df, variant_config):
        """Agrega features por fase fenológica conforme configuração da variante."""
        non_climate_cols = [col for col in df.columns
                            if not any(col.startswith(f"{var}_dec")
                                       for var in self.climate_vars)]
        df_result = df[non_climate_cols].copy()

        for var in self.climate_vars:
            var_cols  = self.get_variable_columns(var)
            if not var_cols:
                continue

            ano1_cols = [c for c in var_cols if 'ano1' in c]
            ano2_cols = [c for c in var_cols if 'ano2' in c]

            # FASE 1: Pré-plantio (ano anterior)
            if variant_config['early']['enabled']:
                n = variant_config['early']['n_decendios']
                if ano1_cols and len(ano1_cols) >= n:
                    early_data = df[ano1_cols[-n:]]
                    cfg = variant_config['early']['stats']
                    if 'mean'   in cfg: df_result[f'{var}_early_mean']   = early_data.mean(axis=1)
                    if 'std'    in cfg: df_result[f'{var}_early_std']    = early_data.std(axis=1)
                    if 'min'    in cfg: df_result[f'{var}_early_min']    = early_data.min(axis=1)
                    if 'max'    in cfg: df_result[f'{var}_early_max']    = early_data.max(axis=1)

            # FASE 2: Florescimento
            fs, fe = variant_config['flowering']['start_dec'], variant_config['flowering']['end_dec']
            flowering_cols = [c for c in ano2_cols
                              if any(f'dec{d}_' in c for d in range(fs, fe + 1))]
            if flowering_cols:
                fd  = df[flowering_cols]
                cfg = variant_config['flowering']['stats']
                if 'mean'   in cfg: df_result[f'{var}_flowering_mean']   = fd.mean(axis=1)
                if 'std'    in cfg: df_result[f'{var}_flowering_std']    = fd.std(axis=1)
                if 'min'    in cfg: df_result[f'{var}_flowering_min']    = fd.min(axis=1)
                if 'max'    in cfg: df_result[f'{var}_flowering_max']    = fd.max(axis=1)
                if 'median' in cfg: df_result[f'{var}_flowering_median'] = fd.median(axis=1)

            # FASE 3: Enchimento de grãos
            gs, ge = variant_config['grain']['start_dec'], variant_config['grain']['end_dec']
            grain_cols = [c for c in ano2_cols
                          if any(f'dec{d}_' in c for d in range(gs, ge + 1))]
            if grain_cols:
                gd  = df[grain_cols]
                cfg = variant_config['grain']['stats']
                if 'mean'   in cfg: df_result[f'{var}_grain_mean']   = gd.mean(axis=1)
                if 'std'    in cfg: df_result[f'{var}_grain_std']    = gd.std(axis=1)
                if 'min'    in cfg: df_result[f'{var}_grain_min']    = gd.min(axis=1)
                if 'max'    in cfg: df_result[f'{var}_grain_max']    = gd.max(axis=1)
                if 'median' in cfg: df_result[f'{var}_grain_median'] = gd.median(axis=1)

            # FASE 4: Maturação (opcional)
            if variant_config['maturation']['enabled']:
                ms, me = variant_config['maturation']['start_dec'], variant_config['maturation']['end_dec']
                mat_cols = [c for c in ano2_cols
                            if any(f'dec{d}_' in c for d in range(ms, me + 1))]
                if mat_cols:
                    md  = df[mat_cols]
                    cfg = variant_config['maturation']['stats']
                    if 'mean' in cfg: df_result[f'{var}_maturation_mean'] = md.mean(axis=1)
                    if 'std'  in cfg: df_result[f'{var}_maturation_std']  = md.std(axis=1)

        return df_result

    # ------------------------------------------------------------------
    def select_features_rf(self, df_train, df_test, n_vars):
        X_train = df_train.drop(columns=[self.target_col])
        y_train = df_train[self.target_col]

        rf = RandomForestRegressor(n_estimators=100, max_depth=8,
                                   min_samples_leaf=5, random_state=42, n_jobs=-1)
        rf.fit(X_train, y_train)

        importances = pd.DataFrame({
            'feature': X_train.columns,
            'importance': rf.feature_importances_
        }).sort_values('importance', ascending=False)

        top_features = importances.head(n_vars)['feature'].tolist()
        return (df_train[top_features + [self.target_col]],
                df_test[top_features  + [self.target_col]],
                top_features)

    # ------------------------------------------------------------------
    def apply_scaling(self, X_train, X_test, scaler_type='robust'):
        if scaler_type == 'robust':
            scaler = RobustScaler()
        elif scaler_type == 'standard':
            scaler = StandardScaler()
        elif scaler_type == 'power':
            scaler = PowerTransformer(method='yeo-johnson', standardize=True)
        else:
            return X_train, X_test
        return scaler.fit_transform(X_train), scaler.transform(X_test)

    # ------------------------------------------------------------------
    def evaluate_with_cv(self, X, y, model, cv=5):
        scores = cross_val_score(model, X, y, cv=cv, scoring='r2', n_jobs=-1)
        return scores.mean(), scores.std()

    # ------------------------------------------------------------------
    def train_and_evaluate(self, df_train, df_test, variant_name,
                           n_features, model_config, scaler_type='robust'):
        X_train = df_train.drop(columns=[self.target_col])
        y_train = df_train[self.target_col]
        X_test  = df_test.drop(columns=[self.target_col])
        y_test  = df_test[self.target_col]

        X_train_s, X_test_s = self.apply_scaling(X_train, X_test, scaler_type)

        mtype  = model_config['type']
        params = model_config['params']

        if mtype == 'gbm':
            model       = GradientBoostingRegressor(**params, random_state=42)
            params_short = (f"{params['n_estimators']}e_d{params['max_depth']}"
                            f"_lr{params['learning_rate']}")
        elif mtype == 'svm':
            model       = SVR(**params)
            params_short = f"SVM_{params.get('kernel','rbf')}_C{params.get('C',1)}"
        elif mtype == 'linear':
            model       = LinearRegression(**params, n_jobs=-1)
            params_short = "Linear_OLS"
        else:   # rf
            model       = RandomForestRegressor(**params, random_state=42, n_jobs=-1)
            params_short = (f"{params['n_estimators']}e_d{params['max_depth']}"
                            f"_msl{params['min_samples_leaf']}")

        cv_mean, cv_std = self.evaluate_with_cv(X_train_s, y_train, model)
        model.fit(X_train_s, y_train)

        y_pred_train = model.predict(X_train_s)
        y_pred_test  = model.predict(X_test_s)

        r2_train = r2_score(y_train, y_pred_train)
        r2_test  = r2_score(y_test,  y_pred_test)

        result = {
            'variant':      variant_name,
            'n_features':   n_features,
            'model':        mtype,
            'scaler':       scaler_type,
            'params_short': params_short,
            'r2_train':     r2_train,
            'r2_test':      r2_test,
            'r2_cv':        cv_mean,
            'cv_std':       cv_std,
            'rmse_test':    np.sqrt(mean_squared_error(y_test, y_pred_test)),
            'mae_test':     mean_absolute_error(y_test, y_pred_test),
            'overfit':      r2_train - r2_test,
        }
        self.results.append(result)
        return result

    # ------------------------------------------------------------------
    def _drop_leakage_cols(self, df):
        """
        Remove colunas que vazam informação sobre o target ou são identificadores.
        Mantém 'ano' (pode ser preditor contextual leve) e 'area_plantada_ha'.
        Ajuste a lista conforme necessidade.
        """
        leakage_cols = [
            'quantidade_produzida_ton',
            'valor_producao_mil_reais',
            'valor_producao_pct',
            'valor_producao_ipca_mil_reais',
            'valor_producao_ipca_mil_reais_ha',
            'area_colhida_ha',
            'area_colhida_pct',
            'area_plantada_pct',
            'fator_correcao_ipca',
            'area_plantada_ha',
            # identificadores (não preditivos)
            'ano', 'cod_ibge', 'municipio', 'cod_meso', 'mesorregiao',
            'latitude', 'longitude',
        ]
        return df.drop(columns=[c for c in leakage_cols if c in df.columns])

    # ------------------------------------------------------------------
    def run_experiments(self):
        print("=" * 120)
        print("DEFINIÇÃO DAS VARIANTES")
        print("=" * 120 + "\n")

        # Remover colunas com leakage antes de qualquer processamento
        train_clean = self._drop_leakage_cols(self.train_df)
        test_clean  = self._drop_leakage_cols(self.test_df)

        # Normalizar dados brutos ANTES da agregação
        train_normalized, test_normalized = self.normalize_raw_data(train_clean, test_clean)

        variants = {
            'v1_original': {
                'name':        'V1: Original (3 dec early, 5-10 flow, 11-15 grain)',
                'early':       {'enabled': True,  'n_decendios': 3, 'stats': ['mean']},
                'flowering':   {'start_dec': 5,  'end_dec': 10, 'stats': ['mean']},
                'grain':       {'start_dec': 11, 'end_dec': 15, 'stats': ['mean']},
                'maturation':  {'enabled': False, 'start_dec': 16, 'end_dec': 18, 'stats': ['mean']},
            },
            'v6_with_variability': {
                'name':        'V6: Com variabilidade (mean + std)',
                'early':       {'enabled': True,  'n_decendios': 3, 'stats': ['mean', 'std']},
                'flowering':   {'start_dec': 5,  'end_dec': 10, 'stats': ['mean', 'std']},
                'grain':       {'start_dec': 11, 'end_dec': 15, 'stats': ['mean', 'std']},
                'maturation':  {'enabled': False, 'start_dec': 16, 'end_dec': 18, 'stats': ['mean']},
            },
            'v8_robust_stats': {
                'name':        'V8: Estatísticas robustas (mean + median)',
                'early':       {'enabled': True,  'n_decendios': 3, 'stats': ['mean']},
                'flowering':   {'start_dec': 5,  'end_dec': 10, 'stats': ['mean', 'median']},
                'grain':       {'start_dec': 11, 'end_dec': 15, 'stats': ['mean', 'median']},
                'maturation':  {'enabled': False, 'start_dec': 16, 'end_dec': 18, 'stats': ['mean']},
            },
            'v9_complete': {
                'name':        'V9: Completo (todas fases + variabilidade)',
                'early':       {'enabled': True,  'n_decendios': 4, 'stats': ['mean', 'std']},
                'flowering':   {'start_dec': 5,  'end_dec': 10, 'stats': ['mean', 'std']},
                'grain':       {'start_dec': 11, 'end_dec': 15, 'stats': ['mean', 'std']},
                'maturation':  {'enabled': True,  'start_dec': 16, 'end_dec': 18, 'stats': ['mean', 'std']},
            },
        }

        for key, cfg in variants.items():
            print(f"📋 {key}: {cfg['name']}")
        print()

        feature_counts = [35, 40, 45, 50]
        scalers        = ['robust', 'standard', 'power']
        model_configs  = [
            {'type': 'rf',     'params': {'n_estimators': 200, 'max_depth': 8,  'min_samples_split': 10, 'min_samples_leaf': 5}},
            {'type': 'rf',     'params': {'n_estimators': 250, 'max_depth': 8,  'min_samples_split': 10, 'min_samples_leaf': 5}},
            {'type': 'rf',     'params': {'n_estimators': 200, 'max_depth': 10, 'min_samples_split': 12, 'min_samples_leaf': 6}},
            {'type': 'gbm',    'params': {'n_estimators': 150, 'max_depth': 5,  'learning_rate': 0.03, 'subsample': 0.85}},
            {'type': 'gbm',    'params': {'n_estimators': 200, 'max_depth': 5,  'learning_rate': 0.02, 'subsample': 0.85}},
            {'type': 'svm',    'params': {'kernel': 'rbf',    'C': 10.0, 'epsilon': 0.1}},
            {'type': 'svm',    'params': {'kernel': 'linear', 'C': 1.0}},
            {'type': 'linear', 'params': {}},
        ]

        print("=" * 120)
        print("EXECUTANDO EXPERIMENTOS COM NORMALIZAÇÃO")
        print("=" * 120 + "\n")

        exp_num  = 0
        best_r2  = -float('inf')

        for var_key, var_config in variants.items():
            print(f"🔬 {var_config['name']}")

            train_agg = self.aggregate_features(train_normalized, var_config)
            test_agg  = self.aggregate_features(test_normalized,  var_config)

            n_features_available = train_agg.shape[1] - 1
            print(f"   Features disponíveis: {n_features_available}")

            for n_vars in feature_counts:
                if n_vars > n_features_available:
                    continue

                df_train_exp, df_test_exp, _ = self.select_features_rf(train_agg, test_agg, n_vars)

                for scaler_type in scalers:
                    for model_config in model_configs:
                        exp_num += 1
                        result   = self.train_and_evaluate(
                            df_train_exp, df_test_exp,
                            var_key, n_vars, model_config, scaler_type
                        )

                        if result['r2_test'] > best_r2:
                            best_r2 = result['r2_test']
                            print(f"   ⭐ [{exp_num:4d}] {model_config['type']} | {scaler_type:8s} | n={n_vars} | "
                                  f"R²Test={result['r2_test']:.4f} | R²CV={result['r2_cv']:.4f} | "
                                  f"Overfit={result['overfit']:.3f}")
                        elif result['r2_test'] > 0.28:
                            print(f"   ✓ [{exp_num:4d}] {model_config['type']} | {scaler_type:8s} | "
                                  f"n={n_vars} | R²={result['r2_test']:.4f}")
            print()

    # ------------------------------------------------------------------
    def display_results(self):
        print("=" * 120)
        print("🏆 TOP 30 RESULTADOS")
        print("=" * 120 + "\n")

        results_df = pd.DataFrame(self.results).sort_values('r2_test', ascending=False)

        print(f"{'#':<4} {'Variante':<22} {'n':<5} {'Modelo':<6} {'Scaler':<10} {'Config':<25} "
              f"{'R²Test':<9} {'R²CV':<9} {'Overfit':<8}")
        print("-" * 130)

        for idx, (_, row) in enumerate(results_df.head(30).iterrows(), 1):
            print(f"{idx:<4} {row['variant']:<22} {row['n_features']:<5} {row['model']:<6} "
                  f"{row['scaler']:<10} {row['params_short']:<25} "
                  f"{row['r2_test']:<9.4f} {row['r2_cv']:<9.4f} {row['overfit']:<8.3f}")

        best = results_df.iloc[0]

        print("\n" + "=" * 120)
        print("🎯 MELHOR CONFIGURAÇÃO COM NORMALIZAÇÃO")
        print("=" * 120)
        print(f"\nVariante: {best['variant']}")
        print(f"Features: {best['n_features']}")
        print(f"Modelo:   {best['model']}")
        print(f"Scaler:   {best['scaler']}")
        print(f"Config:   {best['params_short']}")
        print(f"\n📊 PERFORMANCE:")
        print(f"   R² Teste:    {best['r2_test']:.4f} ⭐")
        print(f"   R² CV:       {best['r2_cv']:.4f} ± {best['cv_std']:.4f}")
        print(f"   R² Treino:   {best['r2_train']:.4f}")
        print(f"   RMSE:        {best['rmse_test']:.2f} kg/ha")
        print(f"   MAE:         {best['mae_test']:.2f} kg/ha")
        print(f"   Overfitting: {best['overfit']:.4f}")

        print(f"\n📊 PERFORMANCE POR TIPO DE NORMALIZAÇÃO:")
        print("-" * 80)
        scaler_summary = results_df.groupby('scaler')['r2_test'].agg(['mean', 'max', 'count']).round(4)
        scaler_summary.columns = ['R² Médio', 'R² Máximo', 'Testes']
        print(scaler_summary.sort_values('R² Máximo', ascending=False))

        print(f"\n📊 PERFORMANCE POR VARIANTE:")
        print("-" * 80)
        variant_summary = results_df.groupby('variant')['r2_test'].agg(['mean', 'max', 'count']).round(4)
        variant_summary.columns = ['R² Médio', 'R² Máximo', 'Testes']
        print(variant_summary.sort_values('R² Máximo', ascending=False))

        print(f"\n📈 COMPARAÇÃO:")
        print(f"   Baseline (sem normalização):  R² = 0.2949")
        print(f"   Com normalização:             R² = {best['r2_test']:.4f}")
        improvement = (best['r2_test'] / 0.2949 - 1) * 100
        sign = '+' if improvement >= 0 else ''
        print(f"   Variação:                     {sign}{improvement:.2f}%")

        return results_df

    # ------------------------------------------------------------------
    def run(self):
        self.load_and_clean_data()
        self.identify_climate_variables()
        self.run_experiments()
        return self.display_results()


# ===============================================================================
# EXECUÇÃO
# ===============================================================================
if __name__ == "__main__":
    tester = CriticalV1VariantsTest(
        parquet_path = PARQUET_PATH,
        target_col   = TARGET_COL,
        year_col     = YEAR_COL,
        train_years  = TRAIN_YEARS,
        test_years   = TEST_YEARS,
    )
    results = tester.run()

    print("\n" + "=" * 120)
    print("✅ TESTE COM NORMALIZAÇÃO MELHORADA CONCLUÍDO!")
    print("=" * 120)

TESTE DE VARIAÇÕES DO CRITICAL_V1 - COM NORMALIZAÇÃO OTIMIZADA

✓ Dataset carregado: (2793, 8226)  |  colunas: ['cod_ibge', 'municipio', 'ano', 'cod_meso', 'mesorregiao', 'latitude', 'longitude', 'area_plantada_ha'] …

✓ Treino (2018–2022): (1995, 8226) | Teste (2023–2024): (798, 8226)

📊 TARGET (rendimento_kg_ha) — ESTATÍSTICAS ORIGINAIS:
   Treino: média=2968.33, std=1050.44, min=0.00, max=5400.00
   Teste : média=3276.66, std=813.62, min=0.00, max=5000.00

✓ 114 variáveis climáticas identificadas: ['AIRMASS', 'ALLSKY_KT', 'ALLSKY_NKT', 'ALLSKY_SFC_LW_DWN', 'ALLSKY_SFC_LW_UP'] …

DEFINIÇÃO DAS VARIANTES

🔧 Aplicando normalização nos dados brutos...
   ✓ 8208 colunas climáticas normalizadas
📋 v1_original: V1: Original (3 dec early, 5-10 flow, 11-15 grain)
📋 v6_with_variability: V6: Com variabilidade (mean + std)
📋 v8_robust_stats: V8: Estatísticas robustas (mean + median)
📋 v9_complete: V9: Completo (todas fases + variabilidade)

EXECUTANDO EXPERIMENTOS COM NORMALIZAÇÃO

🔬 V1: Origina

## area_plantada_ha  + variáveis climáticas --> valor_producao_mil_reais

In [15]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import RobustScaler, StandardScaler, PowerTransformer
from sklearn.model_selection import cross_val_score
import warnings
warnings.filterwarnings('ignore')

# ===============================================================================
# CONFIGURAÇÃO - ajuste apenas estes caminhos se necessário
# ===============================================================================
PARQUET_PATH  = r'C:\Users\bruno\Desktop\Pipeline_TCC\data\processed\dataset_final.parquet'
TARGET_COL    = 'valor_producao_mil_reais'
YEAR_COL      = 'ano'
TRAIN_YEARS   = list(range(2018, 2023))   # 2018 a 2022 inclusive
TEST_YEARS    = [2023, 2024]
# ===============================================================================


class CriticalV1VariantsTest:
    """
    Teste focado em variações do critical_v1 com normalização otimizada.
    Lê um único arquivo parquet e faz o split por ano.
    """

    def __init__(self, parquet_path, target_col, year_col,
                 train_years, test_years):
        self.parquet_path = parquet_path
        self.target_col   = target_col
        self.year_col     = year_col
        self.train_years  = train_years
        self.test_years   = test_years
        self.results      = []

    # ------------------------------------------------------------------
    def load_and_clean_data(self):
        print("=" * 120)
        print("TESTE DE VARIAÇÕES DO CRITICAL_V1 - COM NORMALIZAÇÃO OTIMIZADA")
        print("=" * 120)

        df = pd.read_parquet(self.parquet_path)
        print(f"\n✓ Dataset carregado: {df.shape}  |  colunas: {df.columns.tolist()[:8]} …")

        # Split temporal
        self.train_df = df[df[self.year_col].isin(self.train_years)].dropna().reset_index(drop=True)
        self.test_df  = df[df[self.year_col].isin(self.test_years)].dropna().reset_index(drop=True)

        print(f"\n✓ Treino ({self.train_years[0]}–{self.train_years[-1]}): {self.train_df.shape} "
              f"| Teste ({self.test_years[0]}–{self.test_years[-1]}): {self.test_df.shape}")

        print(f"\n📊 TARGET ({self.target_col}) — ESTATÍSTICAS ORIGINAIS:")
        for label, subset in [("Treino", self.train_df), ("Teste ", self.test_df)]:
            s = subset[self.target_col]
            print(f"   {label}: média={s.mean():.2f}, std={s.std():.2f}, "
                  f"min={s.min():.2f}, max={s.max():.2f}")
        print()

    # ------------------------------------------------------------------
    def identify_climate_variables(self):
        """Detecta prefixos de variáveis climáticas do tipo <var>_dec<N>_ano<M>."""
        all_cols = self.train_df.columns.tolist()
        climate_vars = set()
        for col in all_cols:
            if 'dec' in col and 'ano' in col:
                var_name = col.split('dec')[0].rstrip('_')
                climate_vars.add(var_name)

        self.climate_vars = sorted(list(climate_vars))
        print(f"✓ {len(self.climate_vars)} variáveis climáticas identificadas: {self.climate_vars[:5]} …\n")

    # ------------------------------------------------------------------
    def get_variable_columns(self, var_name):
        return [col for col in self.train_df.columns
                if col.startswith(f"{var_name}_dec")]

    # ------------------------------------------------------------------
    def normalize_raw_data(self, train_df, test_df):
        """Normaliza colunas climáticas brutas ANTES da agregação."""
        print("🔧 Aplicando normalização nos dados brutos...")

        train_normalized = train_df.copy()
        test_normalized  = test_df.copy()

        climate_cols = [col for col in train_df.columns if 'dec' in col and 'ano' in col]

        for var in self.climate_vars:
            var_cols = [col for col in climate_cols if col.startswith(f"{var}_dec")]
            if var_cols:
                scaler = RobustScaler()
                train_normalized[var_cols] = scaler.fit_transform(train_df[var_cols])
                test_normalized[var_cols]  = scaler.transform(test_df[var_cols])

        print(f"   ✓ {len(climate_cols)} colunas climáticas normalizadas")
        return train_normalized, test_normalized

    # ------------------------------------------------------------------
    def aggregate_features(self, df, variant_config):
        """Agrega features por fase fenológica conforme configuração da variante."""
        non_climate_cols = [col for col in df.columns
                            if not any(col.startswith(f"{var}_dec")
                                       for var in self.climate_vars)]
        df_result = df[non_climate_cols].copy()

        for var in self.climate_vars:
            var_cols  = self.get_variable_columns(var)
            if not var_cols:
                continue

            ano1_cols = [c for c in var_cols if 'ano1' in c]
            ano2_cols = [c for c in var_cols if 'ano2' in c]

            # FASE 1: Pré-plantio (ano anterior)
            if variant_config['early']['enabled']:
                n = variant_config['early']['n_decendios']
                if ano1_cols and len(ano1_cols) >= n:
                    early_data = df[ano1_cols[-n:]]
                    cfg = variant_config['early']['stats']
                    if 'mean'   in cfg: df_result[f'{var}_early_mean']   = early_data.mean(axis=1)
                    if 'std'    in cfg: df_result[f'{var}_early_std']    = early_data.std(axis=1)
                    if 'min'    in cfg: df_result[f'{var}_early_min']    = early_data.min(axis=1)
                    if 'max'    in cfg: df_result[f'{var}_early_max']    = early_data.max(axis=1)

            # FASE 2: Florescimento
            fs, fe = variant_config['flowering']['start_dec'], variant_config['flowering']['end_dec']
            flowering_cols = [c for c in ano2_cols
                              if any(f'dec{d}_' in c for d in range(fs, fe + 1))]
            if flowering_cols:
                fd  = df[flowering_cols]
                cfg = variant_config['flowering']['stats']
                if 'mean'   in cfg: df_result[f'{var}_flowering_mean']   = fd.mean(axis=1)
                if 'std'    in cfg: df_result[f'{var}_flowering_std']    = fd.std(axis=1)
                if 'min'    in cfg: df_result[f'{var}_flowering_min']    = fd.min(axis=1)
                if 'max'    in cfg: df_result[f'{var}_flowering_max']    = fd.max(axis=1)
                if 'median' in cfg: df_result[f'{var}_flowering_median'] = fd.median(axis=1)

            # FASE 3: Enchimento de grãos
            gs, ge = variant_config['grain']['start_dec'], variant_config['grain']['end_dec']
            grain_cols = [c for c in ano2_cols
                          if any(f'dec{d}_' in c for d in range(gs, ge + 1))]
            if grain_cols:
                gd  = df[grain_cols]
                cfg = variant_config['grain']['stats']
                if 'mean'   in cfg: df_result[f'{var}_grain_mean']   = gd.mean(axis=1)
                if 'std'    in cfg: df_result[f'{var}_grain_std']    = gd.std(axis=1)
                if 'min'    in cfg: df_result[f'{var}_grain_min']    = gd.min(axis=1)
                if 'max'    in cfg: df_result[f'{var}_grain_max']    = gd.max(axis=1)
                if 'median' in cfg: df_result[f'{var}_grain_median'] = gd.median(axis=1)

            # FASE 4: Maturação (opcional)
            if variant_config['maturation']['enabled']:
                ms, me = variant_config['maturation']['start_dec'], variant_config['maturation']['end_dec']
                mat_cols = [c for c in ano2_cols
                            if any(f'dec{d}_' in c for d in range(ms, me + 1))]
                if mat_cols:
                    md  = df[mat_cols]
                    cfg = variant_config['maturation']['stats']
                    if 'mean' in cfg: df_result[f'{var}_maturation_mean'] = md.mean(axis=1)
                    if 'std'  in cfg: df_result[f'{var}_maturation_std']  = md.std(axis=1)

        return df_result

    # ------------------------------------------------------------------
    def select_features_rf(self, df_train, df_test, n_vars):
        X_train = df_train.drop(columns=[self.target_col])
        y_train = df_train[self.target_col]

        rf = RandomForestRegressor(n_estimators=100, max_depth=8,
                                   min_samples_leaf=5, random_state=42, n_jobs=-1)
        rf.fit(X_train, y_train)

        importances = pd.DataFrame({
            'feature': X_train.columns,
            'importance': rf.feature_importances_
        }).sort_values('importance', ascending=False)

        top_features = importances.head(n_vars)['feature'].tolist()
        return (df_train[top_features + [self.target_col]],
                df_test[top_features  + [self.target_col]],
                top_features)

    # ------------------------------------------------------------------
    def apply_scaling(self, X_train, X_test, scaler_type='robust'):
        if scaler_type == 'robust':
            scaler = RobustScaler()
        elif scaler_type == 'standard':
            scaler = StandardScaler()
        elif scaler_type == 'power':
            scaler = PowerTransformer(method='yeo-johnson', standardize=True)
        else:
            return X_train, X_test
        return scaler.fit_transform(X_train), scaler.transform(X_test)

    # ------------------------------------------------------------------
    def evaluate_with_cv(self, X, y, model, cv=5):
        scores = cross_val_score(model, X, y, cv=cv, scoring='r2', n_jobs=-1)
        return scores.mean(), scores.std()

    # ------------------------------------------------------------------
    def train_and_evaluate(self, df_train, df_test, variant_name,
                           n_features, model_config, scaler_type='robust'):
        X_train = df_train.drop(columns=[self.target_col])
        y_train = df_train[self.target_col]
        X_test  = df_test.drop(columns=[self.target_col])
        y_test  = df_test[self.target_col]

        X_train_s, X_test_s = self.apply_scaling(X_train, X_test, scaler_type)

        mtype  = model_config['type']
        params = model_config['params']

        if mtype == 'gbm':
            model       = GradientBoostingRegressor(**params, random_state=42)
            params_short = (f"{params['n_estimators']}e_d{params['max_depth']}"
                            f"_lr{params['learning_rate']}")
        elif mtype == 'svm':
            model       = SVR(**params)
            params_short = f"SVM_{params.get('kernel','rbf')}_C{params.get('C',1)}"
        elif mtype == 'linear':
            model       = LinearRegression(**params, n_jobs=-1)
            params_short = "Linear_OLS"
        else:   # rf
            model       = RandomForestRegressor(**params, random_state=42, n_jobs=-1)
            params_short = (f"{params['n_estimators']}e_d{params['max_depth']}"
                            f"_msl{params['min_samples_leaf']}")

        cv_mean, cv_std = self.evaluate_with_cv(X_train_s, y_train, model)
        model.fit(X_train_s, y_train)

        y_pred_train = model.predict(X_train_s)
        y_pred_test  = model.predict(X_test_s)

        r2_train = r2_score(y_train, y_pred_train)
        r2_test  = r2_score(y_test,  y_pred_test)

        result = {
            'variant':      variant_name,
            'n_features':   n_features,
            'model':        mtype,
            'scaler':       scaler_type,
            'params_short': params_short,
            'r2_train':     r2_train,
            'r2_test':      r2_test,
            'r2_cv':        cv_mean,
            'cv_std':       cv_std,
            'rmse_test':    np.sqrt(mean_squared_error(y_test, y_pred_test)),
            'mae_test':     mean_absolute_error(y_test, y_pred_test),
            'overfit':      r2_train - r2_test,
        }
        self.results.append(result)
        return result

    # ------------------------------------------------------------------
    def _drop_leakage_cols(self, df):
        """
        Mantém como preditores APENAS:
          - area_plantada_ha
          - variáveis climáticas  (<var>_dec<N>_ano<M>)
        'ano' é descartado para evitar vazamento intra-anual.
        Tudo o mais é descartado (leakage ou identificadores).
        """
        keep = {self.target_col, 'area_plantada_ha'}

        drop_cols = [
            col for col in df.columns
            if col not in keep
            and not (col.startswith(tuple(f"{v}_dec" for v in self.climate_vars)))
        ]
        return df.drop(columns=drop_cols)

    # ------------------------------------------------------------------
    def run_experiments(self):
        print("=" * 120)
        print("DEFINIÇÃO DAS VARIANTES")
        print("=" * 120 + "\n")

        # Manter apenas target, area_plantada_ha e climáticas (ano removido: sem vazamento intra-anual)
        # (identify_climate_variables já foi chamado em run() antes deste método)
        train_clean = self._drop_leakage_cols(self.train_df)
        test_clean  = self._drop_leakage_cols(self.test_df)

        # Normalizar dados brutos ANTES da agregação
        train_normalized, test_normalized = self.normalize_raw_data(train_clean, test_clean)

        variants = {
            'v1_original': {
                'name':        'V1: Original (3 dec early, 5-10 flow, 11-15 grain)',
                'early':       {'enabled': True,  'n_decendios': 3, 'stats': ['mean']},
                'flowering':   {'start_dec': 5,  'end_dec': 10, 'stats': ['mean']},
                'grain':       {'start_dec': 11, 'end_dec': 15, 'stats': ['mean']},
                'maturation':  {'enabled': False, 'start_dec': 16, 'end_dec': 18, 'stats': ['mean']},
            },
            'v6_with_variability': {
                'name':        'V6: Com variabilidade (mean + std)',
                'early':       {'enabled': True,  'n_decendios': 3, 'stats': ['mean', 'std']},
                'flowering':   {'start_dec': 5,  'end_dec': 10, 'stats': ['mean', 'std']},
                'grain':       {'start_dec': 11, 'end_dec': 15, 'stats': ['mean', 'std']},
                'maturation':  {'enabled': False, 'start_dec': 16, 'end_dec': 18, 'stats': ['mean']},
            },
            'v8_robust_stats': {
                'name':        'V8: Estatísticas robustas (mean + median)',
                'early':       {'enabled': True,  'n_decendios': 3, 'stats': ['mean']},
                'flowering':   {'start_dec': 5,  'end_dec': 10, 'stats': ['mean', 'median']},
                'grain':       {'start_dec': 11, 'end_dec': 15, 'stats': ['mean', 'median']},
                'maturation':  {'enabled': False, 'start_dec': 16, 'end_dec': 18, 'stats': ['mean']},
            },
            'v9_complete': {
                'name':        'V9: Completo (todas fases + variabilidade)',
                'early':       {'enabled': True,  'n_decendios': 4, 'stats': ['mean', 'std']},
                'flowering':   {'start_dec': 5,  'end_dec': 10, 'stats': ['mean', 'std']},
                'grain':       {'start_dec': 11, 'end_dec': 15, 'stats': ['mean', 'std']},
                'maturation':  {'enabled': True,  'start_dec': 16, 'end_dec': 18, 'stats': ['mean', 'std']},
            },
        }

        for key, cfg in variants.items():
            print(f"📋 {key}: {cfg['name']}")
        print()

        feature_counts = [35, 40, 45, 50]
        scalers        = ['robust', 'standard', 'power']
        model_configs  = [
            {'type': 'rf',     'params': {'n_estimators': 200, 'max_depth': 8,  'min_samples_split': 10, 'min_samples_leaf': 5}},
            {'type': 'rf',     'params': {'n_estimators': 250, 'max_depth': 8,  'min_samples_split': 10, 'min_samples_leaf': 5}},
            {'type': 'rf',     'params': {'n_estimators': 200, 'max_depth': 10, 'min_samples_split': 12, 'min_samples_leaf': 6}},
            {'type': 'gbm',    'params': {'n_estimators': 150, 'max_depth': 5,  'learning_rate': 0.03, 'subsample': 0.85}},
            {'type': 'gbm',    'params': {'n_estimators': 200, 'max_depth': 5,  'learning_rate': 0.02, 'subsample': 0.85}},
            {'type': 'svm',    'params': {'kernel': 'rbf',    'C': 10.0, 'epsilon': 0.1}},
            {'type': 'svm',    'params': {'kernel': 'linear', 'C': 1.0}},
            {'type': 'linear', 'params': {}},
        ]

        print("=" * 120)
        print("EXECUTANDO EXPERIMENTOS COM NORMALIZAÇÃO")
        print("=" * 120 + "\n")

        exp_num  = 0
        best_r2  = -float('inf')

        for var_key, var_config in variants.items():
            print(f"🔬 {var_config['name']}")

            train_agg = self.aggregate_features(train_normalized, var_config)
            test_agg  = self.aggregate_features(test_normalized,  var_config)

            n_features_available = train_agg.shape[1] - 1
            print(f"   Features disponíveis: {n_features_available}")

            for n_vars in feature_counts:
                if n_vars > n_features_available:
                    continue

                df_train_exp, df_test_exp, _ = self.select_features_rf(train_agg, test_agg, n_vars)

                for scaler_type in scalers:
                    for model_config in model_configs:
                        exp_num += 1
                        result   = self.train_and_evaluate(
                            df_train_exp, df_test_exp,
                            var_key, n_vars, model_config, scaler_type
                        )

                        if result['r2_test'] > best_r2:
                            best_r2 = result['r2_test']
                            print(f"   ⭐ [{exp_num:4d}] {model_config['type']} | {scaler_type:8s} | n={n_vars} | "
                                  f"R²Test={result['r2_test']:.4f} | R²CV={result['r2_cv']:.4f} | "
                                  f"Overfit={result['overfit']:.3f}")
                        elif result['r2_test'] > 0.28:
                            print(f"   ✓ [{exp_num:4d}] {model_config['type']} | {scaler_type:8s} | "
                                  f"n={n_vars} | R²={result['r2_test']:.4f}")
            print()

    # ------------------------------------------------------------------
    def display_results(self):
        print("=" * 120)
        print("🏆 TOP 30 RESULTADOS")
        print("=" * 120 + "\n")

        results_df = pd.DataFrame(self.results).sort_values('r2_test', ascending=False)

        print(f"{'#':<4} {'Variante':<22} {'n':<5} {'Modelo':<6} {'Scaler':<10} {'Config':<25} "
              f"{'R²Test':<9} {'R²CV':<9} {'Overfit':<8}")
        print("-" * 130)

        for idx, (_, row) in enumerate(results_df.head(30).iterrows(), 1):
            print(f"{idx:<4} {row['variant']:<22} {row['n_features']:<5} {row['model']:<6} "
                  f"{row['scaler']:<10} {row['params_short']:<25} "
                  f"{row['r2_test']:<9.4f} {row['r2_cv']:<9.4f} {row['overfit']:<8.3f}")

        best = results_df.iloc[0]

        print("\n" + "=" * 120)
        print("🎯 MELHOR CONFIGURAÇÃO COM NORMALIZAÇÃO")
        print("=" * 120)
        print(f"\nVariante: {best['variant']}")
        print(f"Features: {best['n_features']}")
        print(f"Modelo:   {best['model']}")
        print(f"Scaler:   {best['scaler']}")
        print(f"Config:   {best['params_short']}")
        print(f"\n📊 PERFORMANCE:")
        print(f"   R² Teste:    {best['r2_test']:.4f} ⭐")
        print(f"   R² CV:       {best['r2_cv']:.4f} ± {best['cv_std']:.4f}")
        print(f"   R² Treino:   {best['r2_train']:.4f}")
        print(f"   RMSE:        {best['rmse_test']:.2f} ton")
        print(f"   MAE:         {best['mae_test']:.2f} ton")
        print(f"   Overfitting: {best['overfit']:.4f}")

        print(f"\n📊 PERFORMANCE POR TIPO DE NORMALIZAÇÃO:")
        print("-" * 80)
        scaler_summary = results_df.groupby('scaler')['r2_test'].agg(['mean', 'max', 'count']).round(4)
        scaler_summary.columns = ['R² Médio', 'R² Máximo', 'Testes']
        print(scaler_summary.sort_values('R² Máximo', ascending=False))

        print(f"\n📊 PERFORMANCE POR VARIANTE:")
        print("-" * 80)
        variant_summary = results_df.groupby('variant')['r2_test'].agg(['mean', 'max', 'count']).round(4)
        variant_summary.columns = ['R² Médio', 'R² Máximo', 'Testes']
        print(variant_summary.sort_values('R² Máximo', ascending=False))

        print(f"\n📈 COMPARAÇÃO:")
        print(f"   Baseline (sem normalização):  R² = 0.2949")
        print(f"   Com normalização:             R² = {best['r2_test']:.4f}")
        improvement = (best['r2_test'] / 0.2949 - 1) * 100
        sign = '+' if improvement >= 0 else ''
        print(f"   Variação:                     {sign}{improvement:.2f}%")

        return results_df

    # ------------------------------------------------------------------
    def run(self):
        self.load_and_clean_data()
        self.identify_climate_variables()
        self.run_experiments()
        return self.display_results()


# ===============================================================================
# EXECUÇÃO
# ===============================================================================
if __name__ == "__main__":
    tester = CriticalV1VariantsTest(
        parquet_path = PARQUET_PATH,
        target_col   = TARGET_COL,
        year_col     = YEAR_COL,
        train_years  = TRAIN_YEARS,
        test_years   = TEST_YEARS,
    )
    results = tester.run()

    print("\n" + "=" * 120)
    print("✅ TESTE COM NORMALIZAÇÃO MELHORADA CONCLUÍDO!")
    print("=" * 120)

TESTE DE VARIAÇÕES DO CRITICAL_V1 - COM NORMALIZAÇÃO OTIMIZADA

✓ Dataset carregado: (2793, 8226)  |  colunas: ['cod_ibge', 'municipio', 'ano', 'cod_meso', 'mesorregiao', 'latitude', 'longitude', 'area_plantada_ha'] …

✓ Treino (2018–2022): (1995, 8226) | Teste (2023–2024): (798, 8226)

📊 TARGET (valor_producao_mil_reais) — ESTATÍSTICAS ORIGINAIS:
   Treino: média=79708.98, std=103892.00, min=0.00, max=1167459.00
   Teste : média=107600.34, std=121576.57, min=0.00, max=938903.00

✓ 114 variáveis climáticas identificadas: ['AIRMASS', 'ALLSKY_KT', 'ALLSKY_NKT', 'ALLSKY_SFC_LW_DWN', 'ALLSKY_SFC_LW_UP'] …

DEFINIÇÃO DAS VARIANTES

🔧 Aplicando normalização nos dados brutos...
   ✓ 8208 colunas climáticas normalizadas
📋 v1_original: V1: Original (3 dec early, 5-10 flow, 11-15 grain)
📋 v6_with_variability: V6: Com variabilidade (mean + std)
📋 v8_robust_stats: V8: Estatísticas robustas (mean + median)
📋 v9_complete: V9: Completo (todas fases + variabilidade)

EXECUTANDO EXPERIMENTOS COM NORMAL

## variáveis climáticas --> valor_producao_mil_reais

In [14]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import RobustScaler, StandardScaler, PowerTransformer
from sklearn.model_selection import cross_val_score
import warnings
warnings.filterwarnings('ignore')

# ===============================================================================
# CONFIGURAÇÃO - ajuste apenas estes caminhos se necessário
# ===============================================================================
PARQUET_PATH  = r'C:\Users\bruno\Desktop\Pipeline_TCC\data\processed\dataset_final.parquet'
TARGET_COL    = 'valor_producao_mil_reais'
YEAR_COL      = 'ano'
TRAIN_YEARS   = list(range(2018, 2023))   # 2018 a 2022 inclusive
TEST_YEARS    = [2023, 2024]
# ===============================================================================


class CriticalV1VariantsTest:
    """
    Teste focado em variações do critical_v1 com normalização otimizada.
    Lê um único arquivo parquet e faz o split por ano.
    """

    def __init__(self, parquet_path, target_col, year_col,
                 train_years, test_years):
        self.parquet_path = parquet_path
        self.target_col   = target_col
        self.year_col     = year_col
        self.train_years  = train_years
        self.test_years   = test_years
        self.results      = []

    # ------------------------------------------------------------------
    def load_and_clean_data(self):
        print("=" * 120)
        print("TESTE DE VARIAÇÕES DO CRITICAL_V1 - COM NORMALIZAÇÃO OTIMIZADA")
        print("=" * 120)

        df = pd.read_parquet(self.parquet_path)
        print(f"\n✓ Dataset carregado: {df.shape}  |  colunas: {df.columns.tolist()[:8]} …")

        # Split temporal
        self.train_df = df[df[self.year_col].isin(self.train_years)].dropna().reset_index(drop=True)
        self.test_df  = df[df[self.year_col].isin(self.test_years)].dropna().reset_index(drop=True)

        print(f"\n✓ Treino ({self.train_years[0]}–{self.train_years[-1]}): {self.train_df.shape} "
              f"| Teste ({self.test_years[0]}–{self.test_years[-1]}): {self.test_df.shape}")

        print(f"\n📊 TARGET ({self.target_col}) — ESTATÍSTICAS ORIGINAIS:")
        for label, subset in [("Treino", self.train_df), ("Teste ", self.test_df)]:
            s = subset[self.target_col]
            print(f"   {label}: média={s.mean():.2f}, std={s.std():.2f}, "
                  f"min={s.min():.2f}, max={s.max():.2f}")
        print()

    # ------------------------------------------------------------------
    def identify_climate_variables(self):
        """Detecta prefixos de variáveis climáticas do tipo <var>_dec<N>_ano<M>."""
        all_cols = self.train_df.columns.tolist()
        climate_vars = set()
        for col in all_cols:
            if 'dec' in col and 'ano' in col:
                var_name = col.split('dec')[0].rstrip('_')
                climate_vars.add(var_name)

        self.climate_vars = sorted(list(climate_vars))
        print(f"✓ {len(self.climate_vars)} variáveis climáticas identificadas: {self.climate_vars[:5]} …\n")

    # ------------------------------------------------------------------
    def get_variable_columns(self, var_name):
        return [col for col in self.train_df.columns
                if col.startswith(f"{var_name}_dec")]

    # ------------------------------------------------------------------
    def normalize_raw_data(self, train_df, test_df):
        """Normaliza colunas climáticas brutas ANTES da agregação."""
        print("🔧 Aplicando normalização nos dados brutos...")

        train_normalized = train_df.copy()
        test_normalized  = test_df.copy()

        climate_cols = [col for col in train_df.columns if 'dec' in col and 'ano' in col]

        for var in self.climate_vars:
            var_cols = [col for col in climate_cols if col.startswith(f"{var}_dec")]
            if var_cols:
                scaler = RobustScaler()
                train_normalized[var_cols] = scaler.fit_transform(train_df[var_cols])
                test_normalized[var_cols]  = scaler.transform(test_df[var_cols])

        print(f"   ✓ {len(climate_cols)} colunas climáticas normalizadas")
        return train_normalized, test_normalized

    # ------------------------------------------------------------------
    def aggregate_features(self, df, variant_config):
        """Agrega features por fase fenológica conforme configuração da variante."""
        non_climate_cols = [col for col in df.columns
                            if not any(col.startswith(f"{var}_dec")
                                       for var in self.climate_vars)]
        df_result = df[non_climate_cols].copy()

        for var in self.climate_vars:
            var_cols  = self.get_variable_columns(var)
            if not var_cols:
                continue

            ano1_cols = [c for c in var_cols if 'ano1' in c]
            ano2_cols = [c for c in var_cols if 'ano2' in c]

            # FASE 1: Pré-plantio (ano anterior)
            if variant_config['early']['enabled']:
                n = variant_config['early']['n_decendios']
                if ano1_cols and len(ano1_cols) >= n:
                    early_data = df[ano1_cols[-n:]]
                    cfg = variant_config['early']['stats']
                    if 'mean'   in cfg: df_result[f'{var}_early_mean']   = early_data.mean(axis=1)
                    if 'std'    in cfg: df_result[f'{var}_early_std']    = early_data.std(axis=1)
                    if 'min'    in cfg: df_result[f'{var}_early_min']    = early_data.min(axis=1)
                    if 'max'    in cfg: df_result[f'{var}_early_max']    = early_data.max(axis=1)

            # FASE 2: Florescimento
            fs, fe = variant_config['flowering']['start_dec'], variant_config['flowering']['end_dec']
            flowering_cols = [c for c in ano2_cols
                              if any(f'dec{d}_' in c for d in range(fs, fe + 1))]
            if flowering_cols:
                fd  = df[flowering_cols]
                cfg = variant_config['flowering']['stats']
                if 'mean'   in cfg: df_result[f'{var}_flowering_mean']   = fd.mean(axis=1)
                if 'std'    in cfg: df_result[f'{var}_flowering_std']    = fd.std(axis=1)
                if 'min'    in cfg: df_result[f'{var}_flowering_min']    = fd.min(axis=1)
                if 'max'    in cfg: df_result[f'{var}_flowering_max']    = fd.max(axis=1)
                if 'median' in cfg: df_result[f'{var}_flowering_median'] = fd.median(axis=1)

            # FASE 3: Enchimento de grãos
            gs, ge = variant_config['grain']['start_dec'], variant_config['grain']['end_dec']
            grain_cols = [c for c in ano2_cols
                          if any(f'dec{d}_' in c for d in range(gs, ge + 1))]
            if grain_cols:
                gd  = df[grain_cols]
                cfg = variant_config['grain']['stats']
                if 'mean'   in cfg: df_result[f'{var}_grain_mean']   = gd.mean(axis=1)
                if 'std'    in cfg: df_result[f'{var}_grain_std']    = gd.std(axis=1)
                if 'min'    in cfg: df_result[f'{var}_grain_min']    = gd.min(axis=1)
                if 'max'    in cfg: df_result[f'{var}_grain_max']    = gd.max(axis=1)
                if 'median' in cfg: df_result[f'{var}_grain_median'] = gd.median(axis=1)

            # FASE 4: Maturação (opcional)
            if variant_config['maturation']['enabled']:
                ms, me = variant_config['maturation']['start_dec'], variant_config['maturation']['end_dec']
                mat_cols = [c for c in ano2_cols
                            if any(f'dec{d}_' in c for d in range(ms, me + 1))]
                if mat_cols:
                    md  = df[mat_cols]
                    cfg = variant_config['maturation']['stats']
                    if 'mean' in cfg: df_result[f'{var}_maturation_mean'] = md.mean(axis=1)
                    if 'std'  in cfg: df_result[f'{var}_maturation_std']  = md.std(axis=1)

        return df_result

    # ------------------------------------------------------------------
    def select_features_rf(self, df_train, df_test, n_vars):
        X_train = df_train.drop(columns=[self.target_col])
        y_train = df_train[self.target_col]

        rf = RandomForestRegressor(n_estimators=100, max_depth=8,
                                   min_samples_leaf=5, random_state=42, n_jobs=-1)
        rf.fit(X_train, y_train)

        importances = pd.DataFrame({
            'feature': X_train.columns,
            'importance': rf.feature_importances_
        }).sort_values('importance', ascending=False)

        top_features = importances.head(n_vars)['feature'].tolist()
        return (df_train[top_features + [self.target_col]],
                df_test[top_features  + [self.target_col]],
                top_features)

    # ------------------------------------------------------------------
    def apply_scaling(self, X_train, X_test, scaler_type='robust'):
        if scaler_type == 'robust':
            scaler = RobustScaler()
        elif scaler_type == 'standard':
            scaler = StandardScaler()
        elif scaler_type == 'power':
            scaler = PowerTransformer(method='yeo-johnson', standardize=True)
        else:
            return X_train, X_test
        return scaler.fit_transform(X_train), scaler.transform(X_test)

    # ------------------------------------------------------------------
    def evaluate_with_cv(self, X, y, model, cv=5):
        scores = cross_val_score(model, X, y, cv=cv, scoring='r2', n_jobs=-1)
        return scores.mean(), scores.std()

    # ------------------------------------------------------------------
    def train_and_evaluate(self, df_train, df_test, variant_name,
                           n_features, model_config, scaler_type='robust'):
        X_train = df_train.drop(columns=[self.target_col])
        y_train = df_train[self.target_col]
        X_test  = df_test.drop(columns=[self.target_col])
        y_test  = df_test[self.target_col]

        X_train_s, X_test_s = self.apply_scaling(X_train, X_test, scaler_type)

        mtype  = model_config['type']
        params = model_config['params']

        if mtype == 'gbm':
            model       = GradientBoostingRegressor(**params, random_state=42)
            params_short = (f"{params['n_estimators']}e_d{params['max_depth']}"
                            f"_lr{params['learning_rate']}")
        elif mtype == 'svm':
            model       = SVR(**params)
            params_short = f"SVM_{params.get('kernel','rbf')}_C{params.get('C',1)}"
        elif mtype == 'linear':
            model       = LinearRegression(**params, n_jobs=-1)
            params_short = "Linear_OLS"
        else:   # rf
            model       = RandomForestRegressor(**params, random_state=42, n_jobs=-1)
            params_short = (f"{params['n_estimators']}e_d{params['max_depth']}"
                            f"_msl{params['min_samples_leaf']}")

        cv_mean, cv_std = self.evaluate_with_cv(X_train_s, y_train, model)
        model.fit(X_train_s, y_train)

        y_pred_train = model.predict(X_train_s)
        y_pred_test  = model.predict(X_test_s)

        r2_train = r2_score(y_train, y_pred_train)
        r2_test  = r2_score(y_test,  y_pred_test)

        result = {
            'variant':      variant_name,
            'n_features':   n_features,
            'model':        mtype,
            'scaler':       scaler_type,
            'params_short': params_short,
            'r2_train':     r2_train,
            'r2_test':      r2_test,
            'r2_cv':        cv_mean,
            'cv_std':       cv_std,
            'rmse_test':    np.sqrt(mean_squared_error(y_test, y_pred_test)),
            'mae_test':     mean_absolute_error(y_test, y_pred_test),
            'overfit':      r2_train - r2_test,
        }
        self.results.append(result)
        return result

    # ------------------------------------------------------------------
    def _drop_leakage_cols(self, df):
        """
        Mantém como preditores APENAS:
          - area_plantada_ha
          - variáveis climáticas  (<var>_dec<N>_ano<M>)
        'ano' é descartado para evitar vazamento intra-anual.
        Tudo o mais é descartado (leakage ou identificadores).
        """
        keep = {self.target_col}

        drop_cols = [
            col for col in df.columns
            if col not in keep
            and not (col.startswith(tuple(f"{v}_dec" for v in self.climate_vars)))
        ]
        return df.drop(columns=drop_cols)

    # ------------------------------------------------------------------
    def run_experiments(self):
        print("=" * 120)
        print("DEFINIÇÃO DAS VARIANTES")
        print("=" * 120 + "\n")

        # Manter apenas target, area_plantada_ha e climáticas (ano removido: sem vazamento intra-anual)
        # (identify_climate_variables já foi chamado em run() antes deste método)
        train_clean = self._drop_leakage_cols(self.train_df)
        test_clean  = self._drop_leakage_cols(self.test_df)

        # Normalizar dados brutos ANTES da agregação
        train_normalized, test_normalized = self.normalize_raw_data(train_clean, test_clean)

        variants = {
            'v1_original': {
                'name':        'V1: Original (3 dec early, 5-10 flow, 11-15 grain)',
                'early':       {'enabled': True,  'n_decendios': 3, 'stats': ['mean']},
                'flowering':   {'start_dec': 5,  'end_dec': 10, 'stats': ['mean']},
                'grain':       {'start_dec': 11, 'end_dec': 15, 'stats': ['mean']},
                'maturation':  {'enabled': False, 'start_dec': 16, 'end_dec': 18, 'stats': ['mean']},
            },
            'v6_with_variability': {
                'name':        'V6: Com variabilidade (mean + std)',
                'early':       {'enabled': True,  'n_decendios': 3, 'stats': ['mean', 'std']},
                'flowering':   {'start_dec': 5,  'end_dec': 10, 'stats': ['mean', 'std']},
                'grain':       {'start_dec': 11, 'end_dec': 15, 'stats': ['mean', 'std']},
                'maturation':  {'enabled': False, 'start_dec': 16, 'end_dec': 18, 'stats': ['mean']},
            },
            'v8_robust_stats': {
                'name':        'V8: Estatísticas robustas (mean + median)',
                'early':       {'enabled': True,  'n_decendios': 3, 'stats': ['mean']},
                'flowering':   {'start_dec': 5,  'end_dec': 10, 'stats': ['mean', 'median']},
                'grain':       {'start_dec': 11, 'end_dec': 15, 'stats': ['mean', 'median']},
                'maturation':  {'enabled': False, 'start_dec': 16, 'end_dec': 18, 'stats': ['mean']},
            },
            'v9_complete': {
                'name':        'V9: Completo (todas fases + variabilidade)',
                'early':       {'enabled': True,  'n_decendios': 4, 'stats': ['mean', 'std']},
                'flowering':   {'start_dec': 5,  'end_dec': 10, 'stats': ['mean', 'std']},
                'grain':       {'start_dec': 11, 'end_dec': 15, 'stats': ['mean', 'std']},
                'maturation':  {'enabled': True,  'start_dec': 16, 'end_dec': 18, 'stats': ['mean', 'std']},
            },
        }

        for key, cfg in variants.items():
            print(f"📋 {key}: {cfg['name']}")
        print()

        feature_counts = [35, 40, 45, 50]
        scalers        = ['robust', 'standard', 'power']
        model_configs  = [
            {'type': 'rf',     'params': {'n_estimators': 200, 'max_depth': 8,  'min_samples_split': 10, 'min_samples_leaf': 5}},
            {'type': 'rf',     'params': {'n_estimators': 250, 'max_depth': 8,  'min_samples_split': 10, 'min_samples_leaf': 5}},
            {'type': 'rf',     'params': {'n_estimators': 200, 'max_depth': 10, 'min_samples_split': 12, 'min_samples_leaf': 6}},
            {'type': 'gbm',    'params': {'n_estimators': 150, 'max_depth': 5,  'learning_rate': 0.03, 'subsample': 0.85}},
            {'type': 'gbm',    'params': {'n_estimators': 200, 'max_depth': 5,  'learning_rate': 0.02, 'subsample': 0.85}},
            {'type': 'svm',    'params': {'kernel': 'rbf',    'C': 10.0, 'epsilon': 0.1}},
            {'type': 'svm',    'params': {'kernel': 'linear', 'C': 1.0}},
            {'type': 'linear', 'params': {}},
        ]

        print("=" * 120)
        print("EXECUTANDO EXPERIMENTOS COM NORMALIZAÇÃO")
        print("=" * 120 + "\n")

        exp_num  = 0
        best_r2  = -float('inf')

        for var_key, var_config in variants.items():
            print(f"🔬 {var_config['name']}")

            train_agg = self.aggregate_features(train_normalized, var_config)
            test_agg  = self.aggregate_features(test_normalized,  var_config)

            n_features_available = train_agg.shape[1] - 1
            print(f"   Features disponíveis: {n_features_available}")

            for n_vars in feature_counts:
                if n_vars > n_features_available:
                    continue

                df_train_exp, df_test_exp, _ = self.select_features_rf(train_agg, test_agg, n_vars)

                for scaler_type in scalers:
                    for model_config in model_configs:
                        exp_num += 1
                        result   = self.train_and_evaluate(
                            df_train_exp, df_test_exp,
                            var_key, n_vars, model_config, scaler_type
                        )

                        if result['r2_test'] > best_r2:
                            best_r2 = result['r2_test']
                            print(f"   ⭐ [{exp_num:4d}] {model_config['type']} | {scaler_type:8s} | n={n_vars} | "
                                  f"R²Test={result['r2_test']:.4f} | R²CV={result['r2_cv']:.4f} | "
                                  f"Overfit={result['overfit']:.3f}")
                        elif result['r2_test'] > 0.28:
                            print(f"   ✓ [{exp_num:4d}] {model_config['type']} | {scaler_type:8s} | "
                                  f"n={n_vars} | R²={result['r2_test']:.4f}")
            print()

    # ------------------------------------------------------------------
    def display_results(self):
        print("=" * 120)
        print("🏆 TOP 30 RESULTADOS")
        print("=" * 120 + "\n")

        results_df = pd.DataFrame(self.results).sort_values('r2_test', ascending=False)

        print(f"{'#':<4} {'Variante':<22} {'n':<5} {'Modelo':<6} {'Scaler':<10} {'Config':<25} "
              f"{'R²Test':<9} {'R²CV':<9} {'Overfit':<8}")
        print("-" * 130)

        for idx, (_, row) in enumerate(results_df.head(30).iterrows(), 1):
            print(f"{idx:<4} {row['variant']:<22} {row['n_features']:<5} {row['model']:<6} "
                  f"{row['scaler']:<10} {row['params_short']:<25} "
                  f"{row['r2_test']:<9.4f} {row['r2_cv']:<9.4f} {row['overfit']:<8.3f}")

        best = results_df.iloc[0]

        print("\n" + "=" * 120)
        print("🎯 MELHOR CONFIGURAÇÃO COM NORMALIZAÇÃO")
        print("=" * 120)
        print(f"\nVariante: {best['variant']}")
        print(f"Features: {best['n_features']}")
        print(f"Modelo:   {best['model']}")
        print(f"Scaler:   {best['scaler']}")
        print(f"Config:   {best['params_short']}")
        print(f"\n📊 PERFORMANCE:")
        print(f"   R² Teste:    {best['r2_test']:.4f} ⭐")
        print(f"   R² CV:       {best['r2_cv']:.4f} ± {best['cv_std']:.4f}")
        print(f"   R² Treino:   {best['r2_train']:.4f}")
        print(f"   RMSE:        {best['rmse_test']:.2f} ton")
        print(f"   MAE:         {best['mae_test']:.2f} ton")
        print(f"   Overfitting: {best['overfit']:.4f}")

        print(f"\n📊 PERFORMANCE POR TIPO DE NORMALIZAÇÃO:")
        print("-" * 80)
        scaler_summary = results_df.groupby('scaler')['r2_test'].agg(['mean', 'max', 'count']).round(4)
        scaler_summary.columns = ['R² Médio', 'R² Máximo', 'Testes']
        print(scaler_summary.sort_values('R² Máximo', ascending=False))

        print(f"\n📊 PERFORMANCE POR VARIANTE:")
        print("-" * 80)
        variant_summary = results_df.groupby('variant')['r2_test'].agg(['mean', 'max', 'count']).round(4)
        variant_summary.columns = ['R² Médio', 'R² Máximo', 'Testes']
        print(variant_summary.sort_values('R² Máximo', ascending=False))

        print(f"\n📈 COMPARAÇÃO:")
        print(f"   Baseline (sem normalização):  R² = 0.2949")
        print(f"   Com normalização:             R² = {best['r2_test']:.4f}")
        improvement = (best['r2_test'] / 0.2949 - 1) * 100
        sign = '+' if improvement >= 0 else ''
        print(f"   Variação:                     {sign}{improvement:.2f}%")

        return results_df

    # ------------------------------------------------------------------
    def run(self):
        self.load_and_clean_data()
        self.identify_climate_variables()
        self.run_experiments()
        return self.display_results()


# ===============================================================================
# EXECUÇÃO
# ===============================================================================
if __name__ == "__main__":
    tester = CriticalV1VariantsTest(
        parquet_path = PARQUET_PATH,
        target_col   = TARGET_COL,
        year_col     = YEAR_COL,
        train_years  = TRAIN_YEARS,
        test_years   = TEST_YEARS,
    )
    results = tester.run()

    print("\n" + "=" * 120)
    print("✅ TESTE COM NORMALIZAÇÃO MELHORADA CONCLUÍDO!")
    print("=" * 120)

TESTE DE VARIAÇÕES DO CRITICAL_V1 - COM NORMALIZAÇÃO OTIMIZADA

✓ Dataset carregado: (2793, 8226)  |  colunas: ['cod_ibge', 'municipio', 'ano', 'cod_meso', 'mesorregiao', 'latitude', 'longitude', 'area_plantada_ha'] …

✓ Treino (2018–2022): (1995, 8226) | Teste (2023–2024): (798, 8226)

📊 TARGET (valor_producao_mil_reais) — ESTATÍSTICAS ORIGINAIS:
   Treino: média=79708.98, std=103892.00, min=0.00, max=1167459.00
   Teste : média=107600.34, std=121576.57, min=0.00, max=938903.00

✓ 114 variáveis climáticas identificadas: ['AIRMASS', 'ALLSKY_KT', 'ALLSKY_NKT', 'ALLSKY_SFC_LW_DWN', 'ALLSKY_SFC_LW_UP'] …

DEFINIÇÃO DAS VARIANTES

🔧 Aplicando normalização nos dados brutos...
   ✓ 8208 colunas climáticas normalizadas
📋 v1_original: V1: Original (3 dec early, 5-10 flow, 11-15 grain)
📋 v6_with_variability: V6: Com variabilidade (mean + std)
📋 v8_robust_stats: V8: Estatísticas robustas (mean + median)
📋 v9_complete: V9: Completo (todas fases + variabilidade)

EXECUTANDO EXPERIMENTOS COM NORMAL